# 20 — Spatial recorded-count experiments (V3)

Research experiments for the preregistered spatial protocol (`workflow/spatial-model-protocol-v3-draft.md` + `workflow/spatial-model-protocol-v3-addendum.md`).
Target: **SELECTED+LINKED recorded database counts** per qualified DOH segment x local day (primary) or x 6 h block (secondary) — not all physical accidents, not a per-vehicle probability.

Order of work: verify manager pins and the accepted notebook-19 FUNCTIONS/data → persist the immutable experiment lock → four expanding temporal folds (2022H1..2023H2, 30-day label embargo, train-only zero sampling for models only, exact full-calendar baselines) → full-population streamed OOF evaluation → frozen selection + exploratory 7-day block bootstrap → previously seen 2024/25 diagnostics (selected policy + strongest baseline only) → fixed geometry-only spatial-transfer diagnostic → reports/manifest.

Run headless with `.venv/bin/python backend/scripts/run_spatial_experiments.py --output-dir backend/rerun-spatial-model-<id>`. No serving, colours or API here.


In [ ]:
# PARAMETERS: roots come from the runner environment (ROADAWARE_BACKEND_ROOT / ROADAWARE_V3_OUTPUT_ROOT); interactive default = this repo's backend/.
import os
from pathlib import Path


def _default_backend(c):
    for p in (c, *c.parents):
        if (p / 'config' / 'spatial_experiment_v3.json').is_file():
            return p
        if (p / 'backend' / 'config' / 'spatial_experiment_v3.json').is_file():
            return p / 'backend'
    return c


BACKEND = Path(os.environ.get('ROADAWARE_BACKEND_ROOT') or _default_backend(Path.cwd())).resolve()
OUTPUT_ROOT = Path(os.environ.get('ROADAWARE_V3_OUTPUT_ROOT') or BACKEND).resolve()
CONFIG_PATH = BACKEND / 'config' / 'spatial_experiment_v3.json'
print('backend', BACKEND, '| output root', OUTPUT_ROOT)


In [ ]:
# FUNCTIONS (1/6): errors, hashing, strict JSON, output guards, manager-pin verification, the hash-verified notebook-19 FUNCTIONS loader and the
# machine-readable experiment lock. Pure logic; the tests exec these cells. Notebook-19 helpers are bound into H19 (never re-implemented here).
import datetime as dt
import hashlib
import importlib
import importlib.util
import json
import math
import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

NL = chr(10)
UNITS = ('daily', '6h')
H19 = {}
H19_REQUIRED = ('load_prepared_context', 'build_feature_rows', 'feature_columns', 'validate_columns', 'population_axes', 'population_size', 'sparse_index',
                'lookup_counts', 'iter_population_batches', 'keys_from_positions', 'read_numeric_csv', 'frames_identical', 'RequestError', 'PreparationError',
                'WX_DAILY', 'WX_BLOCK', 'AADT', 'STATIC_FEATURES')


class ExperimentError(RuntimeError):
    pass


def utcnow():
    return dt.datetime.now(dt.timezone.utc).isoformat()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()


def atomic_write_bytes(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp-' + str(os.getpid()))
    with open(tmp, 'wb') as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def jsonable(o):
    # numpy scalars to Python, non-finite floats to None, so json.dumps(allow_nan=False) stays strict
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [jsonable(v) for v in (sorted(o) if isinstance(o, set) else o)]
    if isinstance(o, np.ndarray):
        return [jsonable(v) for v in o.tolist()]
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if math.isfinite(float(o)) else None
    if isinstance(o, (pd.Timestamp, dt.datetime, dt.date)):
        return o.isoformat()
    if isinstance(o, Path):
        return str(o)
    if o is pd.NA or o is pd.NaT:
        return None
    return o


def strict_json_bytes(obj):
    return (json.dumps(jsonable(obj), indent=2, ensure_ascii=False, allow_nan=False) + NL).encode('utf-8')


def canonical_sha(obj):
    return sha256_bytes(json.dumps(jsonable(obj), sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False).encode('utf-8'))


def atomic_write_json(path, obj):
    atomic_write_bytes(path, strict_json_bytes(obj))


def read_strict_json(path):
    def bad(c):
        raise ExperimentError('non-finite JSON constant ' + c + ' in ' + str(path))
    return json.loads(Path(path).read_text('utf-8'), parse_constant=bad)


def csv_bytes(df):
    num = df.select_dtypes(include=[np.number])
    if len(num.columns) and np.isinf(num.to_numpy(dtype=float)).any():
        raise ExperimentError('refusing to write infinite values to CSV')
    return df.to_csv(index=False, lineterminator=NL).encode('utf-8')


def module_version(name):
    if importlib.util.find_spec(name) is None:
        return None
    return str(getattr(importlib.import_module(name), '__version__', 'unknown'))


def require_modules(names, pins):
    missing = [n for n in names if importlib.util.find_spec(n) is None]
    if missing:
        raise ExperimentError('missing Python package(s) ' + ', '.join(missing) + '; ask Codex to install ' + ' '.join(pins.get(n, n) for n in missing))
    sk = module_version('sklearn')
    if 'sklearn' in names and sk != '1.6.1':
        raise ExperimentError('scikit-learn ' + str(sk) + ' != locked 1.6.1 (estimator defaults could differ)')
    return True


def environment():
    return {'python': platform.python_version(), 'platform': platform.platform(),
            'dependencies': {m: module_version(m) for m in ('pandas', 'numpy', 'sklearn', 'joblib', 'threadpoolctl', 'shapely', 'pyproj', 'matplotlib')}}


# ---------- output guards ----------
def output_paths(out_root, cfg):
    out_root = Path(out_root)
    rep, data = out_root / cfg['outputs']['report_dir'], out_root / cfg['outputs']['data_dir']
    return {'out_root': out_root, 'report_dir': rep, 'figure_dir': rep / 'figures', 'data_dir': data, 'array_dir': data / 'arrays', 'model_dir': data / 'models'}


def _same(a, b):
    try:
        return os.path.samefile(a, b)
    except OSError:
        return False


def _inside(child, parent):
    c, p = Path(child).resolve(), Path(parent).resolve()
    if c == p or p in c.parents:
        return True
    return p.exists() and any(_same(a, p) for a in (c, *c.parents) if a.exists())


def input_rels(cfg):
    I = cfg['inputs']
    return [v['path'] for v in I.values() if isinstance(v, dict) and 'path' in v] + [I['prepared_data_dir'], cfg['pins']['path']]


def protected_rels(cfg):
    return sorted(set(cfg['protected_paths']) | set(input_rels(cfg)))


def guarded_paths(backend, cfg):
    backend = Path(backend)
    return [backend / r for r in protected_rels(cfg)] + [backend / cfg['notebook'], backend / 'config' / 'spatial_experiment_v3.json']


def check_outputs(paths, backend, cfg):
    """Call before any mkdir/write: report/data dirs must not overlap (either direction, symlinks/aliases resolved) any protected path, pinned
    input, protocol, source notebook or config, must not overlap each other, and must be NEW."""
    hits = []
    for label in ('report_dir', 'data_dir'):
        for p in guarded_paths(backend, cfg):
            if _inside(paths[label], p) or _inside(p, paths[label]):
                hits.append(label + ' ' + str(paths[label]) + ' overlaps protected ' + str(p))
    if _inside(paths['report_dir'], paths['data_dir']) or _inside(paths['data_dir'], paths['report_dir']):
        hits.append('report and data dirs overlap')
    if hits:
        raise ExperimentError('refusing output root (nothing written): ' + '; '.join(hits))
    used = [str(paths[k]) for k in ('report_dir', 'data_dir') if os.path.lexists(paths[k])]
    if used:
        raise ExperimentError('output already exists in ' + ', '.join(used) + '; use a NEW root, e.g. backend/rerun-spatial-model-<id>')
    return True


def check_report_file(rel, cfg):
    if Path(rel).suffix.lower() not in cfg['report_allowed_suffixes'] or str(rel).endswith('.gz'):
        raise ExperimentError('refusing to write ' + str(rel) + ' into the report dir (aggregated csv/png/md/json only)')
    return True


def hash_paths(root, rels):
    root, out = Path(root), {}
    for rel in rels:
        p = root / rel
        if p.is_file():
            out[Path(rel).as_posix()] = sha256_file(p)
        elif p.is_dir():
            for q in sorted(p.rglob('*')):
                if q.is_file():
                    out[(Path(rel) / q.relative_to(p)).as_posix()] = sha256_file(q)
        else:
            out[Path(rel).as_posix()] = None
    return out


# ---------- manager pins and inputs ----------
def verify_pinned(path, expected, label):
    path = Path(path)
    if not expected:
        raise ExperimentError(label + ': no sha256 pin (fail closed)')
    if not path.is_file():
        raise ExperimentError(label + ': missing input ' + str(path))
    got = sha256_file(path)
    if got != expected:
        raise ExperimentError(label + ': sha256 ' + got[:12] + ' != pinned ' + str(expected)[:12] + ' (' + str(path) + '); input changed or tampered')
    return got


def load_pins(backend, cfg):
    """Manager pins file; every value the config expects must equal it (two independent records of the same lock inputs)."""
    p = Path(backend) / cfg['pins']['path']
    pins = read_strict_json(p)
    if pins.get('registered_before_spatial_model_scores') is not True:
        raise ExperimentError('pins are not registered before spatial model scores')
    e, I = cfg['pins']['expect'], cfg['inputs']
    pairs = [('preparation_manifest_sha256', pins['files'].get(I['preparation_manifest']['pins_key'])),
             ('notebook19_functions_sha256', pins.get('notebook19_functions_sha256')),
             ('protocol_draft_sha256', pins['files'].get(I['protocol_draft']['pins_key'])),
             ('protocol_addendum_sha256', pins['files'].get(I['protocol_addendum']['pins_key'])),
             ('transfer_preregistration_sha256', pins['files'].get(I['transfer_preregistration']['pins_key']))]
    bad = [k for k, v in pairs if v != e[k]]
    if bad:
        raise ExperimentError('manager pins differ from the config expectation: ' + str(bad))
    return pins, sha256_file(p)


def verify_inputs(backend, cfg, pins):
    """Hash-verify every pinned input and every prepared data file; the accepted 19 manifest must record complete status, the pinned FUNCTIONS
    and code-cell hashes and exactly the pinned data hashes. Returns (rows DataFrame, docs)."""
    backend, I, rows, docs = Path(backend), cfg['inputs'], [], {}
    for key, v in I.items():
        if not isinstance(v, dict) or 'pins_key' not in v:
            continue
        h = verify_pinned(backend / v['path'], pins['files'].get(v['pins_key']), key)
        rows.append({'input': key, 'path': v['path'], 'sha256': h, 'rule': 'manager pin', 'status': 'pass'})
        if v['path'].endswith('.json'):
            docs[key] = read_strict_json(backend / v['path'])
    m = docs['preparation_manifest']
    if m.get('status') != I['preparation_manifest']['expect_status']:
        raise ExperimentError('notebook-19 manifest status is ' + str(m.get('status')) + ', not complete')
    if m.get('functions_cells_sha256') != pins['notebook19_functions_sha256']:
        raise ExperimentError('notebook-19 manifest FUNCTIONS sha differs from the manager pin')
    if m.get('code_cells_sha256') != pins['notebook19_code_cells_sha256']:
        raise ExperimentError('notebook-19 manifest code-cell hashes differ from the manager pins')
    if m.get('data_files_gitignored') != pins['prepared_data_hashes']:
        raise ExperimentError('notebook-19 manifest data_files_gitignored differs from the manager pins')
    dd = backend / I['prepared_data_dir']
    for name, h in sorted(pins['prepared_data_hashes'].items()):
        rows.append({'input': 'prepared:' + name, 'path': (Path(I['prepared_data_dir']) / name).as_posix(), 'sha256': verify_pinned(dd / name, h, 'prepared ' + name),
                     'rule': 'manager pin == 19 manifest', 'status': 'pass'})
    return pd.DataFrame(rows), docs


# ---------- verified notebook-19 FUNCTIONS ----------
def notebook_cells(nb_path):
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    out = []
    for i, c in enumerate(nb['cells']):
        if c['cell_type'] == 'code':
            out.append((i, str(c.get('id', '')), ''.join(c['source']) if isinstance(c['source'], list) else c['source']))
    return out


def load_nb19_functions(nb_path, pins, expected_cells=6):
    """exec ONLY the six notebook-19 '# FUNCTIONS' cells after (a) every code-cell source sha equals the manager pin (index:id keys, the
    executed 19's own convention) and (b) the '\\n\\n'-joined FUNCTIONS sha equals notebook19_functions_sha256. Binds them into H19."""
    cells = notebook_cells(nb_path)
    got = {str(i) + ':' + cid: sha256_bytes(src.encode('utf-8')) for i, cid, src in cells}
    if got != pins['notebook19_code_cells_sha256']:
        diff = sorted(k for k in set(got) | set(pins['notebook19_code_cells_sha256']) if got.get(k) != pins['notebook19_code_cells_sha256'].get(k))
        raise ExperimentError('notebook-19 code cells differ from the manager pins: ' + str(diff[:5]))
    funcs = [src for _, _, src in cells if src.lstrip().startswith('# FUNCTIONS')]
    sha = sha256_bytes('\n\n'.join(funcs).encode('utf-8'))
    if len(funcs) != int(expected_cells) or sha != pins['notebook19_functions_sha256']:
        raise ExperimentError('notebook-19 FUNCTIONS sha ' + sha[:12] + ' (' + str(len(funcs)) + ' cells) != pinned ' + str(pins['notebook19_functions_sha256'])[:12])
    ns = {'__name__': 'nb19_functions'}
    for i, s in enumerate(funcs):
        exec(compile(s, 'notebook19:FUNCTIONS[' + str(i) + ']', 'exec'), ns)
    miss = [n for n in H19_REQUIRED if n not in ns]
    if miss:
        raise ExperimentError('notebook-19 helpers missing: ' + str(miss))
    H19.clear()
    H19.update(ns)
    return ns, sha, got


def check_feature_columns(settings, pins):
    """Feature columns from the pinned 19 builder must equal the manager pins; categoricals route_code/time_block only; keys/labels refused."""
    out = {}
    for unit in UNITS:
        out[unit] = {}
        for var in ('calendar_geometry', 'calendar_geometry_context'):
            cols = H19['feature_columns'](settings, unit, var)
            if cols != pins['feature_columns'][unit][var]:
                raise ExperimentError('feature columns ' + unit + '/' + var + ' differ from the manager pins')
            H19['validate_columns'](cols, settings)
            if set(cols) & {'segment_id', 'target_date', 'y_count', 'oid', 'event_lat', 'event_lon'}:
                raise ExperimentError('key/label/event column offered as a predictor')
            out[unit][var] = cols
        core, ctxv = out[unit]['calendar_geometry'], out[unit]['calendar_geometry_context']
        if ctxv[:len(core)] != core:
            raise ExperimentError('core variant is not the leading prefix of the context variant (single-build slicing would be invalid)')
    return out


def build_lock(cfg, cfg_sha, pins, pins_sha, input_rows, nb19, feature_cols, fold_rows, budget, env):
    """The immutable experiment lock (persisted before any fit or score; its file sha is stamped into every result)."""
    keys = ('time', 'sampling', 'baselines', 'candidates', 'metrics', 'selection', 'bootstrap', 'transfer', 'resources')
    return {'kind': 'roadaware_spatial_experiment_lock', 'version': cfg['version'], 'created_utc': utcnow(),
            'registered_before_fit_or_scores': True, 'label_meaning': cfg['label_meaning'], 'config_sha256': cfg_sha,
            'manager_pins': pins, 'manager_pins_sha256': pins_sha, 'inputs': input_rows,
            'protocol_sha256': {'draft': pins['files'][cfg['inputs']['protocol_draft']['pins_key']], 'addendum': pins['files'][cfg['inputs']['protocol_addendum']['pins_key']]},
            'notebook19': nb19, 'feature_columns': feature_cols, 'categorical_columns': cfg['candidates']['categorical_columns'],
            'feature_builder': 'notebook-19 build_feature_rows(load_prepared_context(prepared_data_dir, data_files_gitignored), keys, unit, calendar_geometry_context); core = leading prefix slice',
            'numeric_csv_parsing': "float_precision='round_trip' (notebook-19 read_numeric_csv)",
            'folds': fold_rows, 'artifact_budget_estimate': budget, 'environment': env,
            **{k: cfg[k] for k in keys}}


def write_lock(lock, paths):
    b = strict_json_bytes(lock)
    for d in ('data_dir', 'report_dir'):
        p = paths[d] / 'experiment_lock.json'
        if os.path.lexists(p):
            raise ExperimentError('experiment lock already exists (immutable): ' + str(p))
        atomic_write_bytes(p, b)
    return sha256_bytes(b)


def verify_lock(paths, lock_sha):
    for d in ('data_dir', 'report_dir'):
        verify_pinned(paths[d] / 'experiment_lock.json', lock_sha, 'experiment lock (' + d + ')')
    return True


In [ ]:
# FUNCTIONS (2/6): issue times, target-END label embargo, fold table, sparse-position decomposition (date -> segment -> block) and the
# train-only stratified zero sampling (all positives + min(32, Nzero) zeros per segment x block, SHA-derived stratum seeds, inverse-inclusion weights).
def unit_blocks(ax, unit):
    if unit not in UNITS:
        raise ExperimentError('unit must be daily|6h')
    return len(ax['blocks']) if unit == '6h' else 1


def block_end_hours(ax, unit):
    """Local wall-clock end of each block relative to the target date's 00:00 (daily: 24 h; 6 h: '00-06' -> 6 ... '18-24' -> 24)."""
    if unit == 'daily':
        return [24]
    return [int(b.split('-')[1]) for b in ax['blocks']]


def issue_time(target_date, cfg):
    """Naive Asia/Bangkok wall time: target day - issue_days_before at issue_hour_local (18:00 the previous day)."""
    t = cfg['time']
    return pd.Timestamp(target_date) - pd.Timedelta(days=int(t['issue_days_before_target'])) + pd.Timedelta(hours=int(t['issue_hour_local']))


def training_cutoff(first_target_date, cfg):
    """A training target is usable only when its END <= first issue of the evaluation period - label_latency_days (research assumption)."""
    return issue_time(first_target_date, cfg) - pd.Timedelta(days=int(cfg['time']['label_latency_days']))


def train_dates_per_block(ax, unit, cutoff):
    """T_b = number of axis dates (from the axis start) whose block-b END <= cutoff. Training always starts at the axis start (2021-01-01)."""
    start = pd.Timestamp(ax['dates'][0])
    out = []
    for h in block_end_hours(ax, unit):
        last = (pd.Timestamp(cutoff) - pd.Timedelta(hours=h)).normalize()
        out.append(int(min(max((last - start).days + 1, 0), len(ax['dates']))))
    return np.asarray(out, np.int64)


def last_train_target_end(ax, unit, T):
    ends = []
    for b, h in enumerate(block_end_hours(ax, unit)):
        if T[b] > 0:
            ends.append(pd.Timestamp(ax['dates'][T[b] - 1]) + pd.Timedelta(hours=h))
    return max(ends) if ends else None


def fold_rows(cfg, ax_by_unit):
    rows = []
    for unit in UNITS:
        ax = ax_by_unit[unit]
        for f in cfg['time']['folds'] + [{'fold_id': 'benchmark_2024', 'start': cfg['time']['benchmark']['fit_issue_target_date'], 'end': None}]:
            cut = training_cutoff(f['start'], cfg)
            T = train_dates_per_block(ax, unit, cut)
            end = last_train_target_end(ax, unit, T)
            if end is None or end > cut or ax['date_pos'][f['start']] <= int(T.max()) - 1:
                raise ExperimentError('embargo violated or empty training window for ' + unit + ' ' + f['fold_id'])
            rows.append({'unit': unit, 'fold_id': f['fold_id'], 'eval_start': f['start'], 'eval_end': f['end'], 'first_issue_local': str(issue_time(f['start'], cfg)),
                         'train_cutoff_local': str(cut), 'last_train_target_end_local': str(end), 'train_dates_per_block': [int(t) for t in T],
                         'last_train_date_per_block': [ax['dates'][t - 1] for t in T]})
    return rows


def decompose(ax, unit, pos):
    nb, S = unit_blocks(ax, unit), len(ax['segments'])
    pos = np.asarray(pos, np.int64)
    return pos // (S * nb), (pos // nb) % S, pos % nb


def compose(ax, unit, d, s, b):
    nb, S = unit_blocks(ax, unit), len(ax['segments'])
    return (np.asarray(d, np.int64) * S + np.asarray(s, np.int64)) * nb + np.asarray(b, np.int64)


def check_sparse(sp, ax, unit):
    """The 19 sparse index must carry the unit/size/axis bound contract for exactly this axis."""
    ev = {'segments': len(ax['segments']), 'first_segment': ax['segments'][0], 'last_segment': ax['segments'][-1], 'dates': len(ax['dates']),
          'first_date': ax['dates'][0], 'last_date': ax['dates'][-1], 'blocks': list(ax['blocks'])}
    if sp.get('unit') != unit or int(sp.get('size', -1)) != H19['population_size'](ax, unit) or sp.get('axis') != ev:
        raise ExperimentError(unit + ' sparse index lacks or mismatches the 19 unit/size/axis bound contract')
    return True


def train_counts(ax, unit, sp, T):
    """Positive training cells (date < T_b for its block) from the sparse index: (d, s, b, y)."""
    check_sparse(sp, ax, unit)
    d, s, b = decompose(ax, unit, sp['pos'])
    m = d < np.asarray(T)[b]
    return d[m], s[m], b[m], sp['y'][m].astype(np.int64)


def stratum_seed(seed, unit, fold_id, segment_id, block):
    return int(sha256_bytes((str(seed) + '|' + unit + '|' + str(fold_id) + '|' + str(segment_id) + '|' + str(block)).encode('utf-8'))[:16], 16)


def sample_training(ax, unit, sp, T, fold_id, cfg):
    """All positive cells + min(K, Nzero) zero cells drawn uniformly WITHOUT replacement within each segment x block stratum over that stratum's
    eligible training dates. Weights: positive 1, sampled zero Nzero/Nsampled. Returns (positions sorted date->segment->block, y, w, summary).
    Used ONLY for model training; baselines and evaluation use the exact full population."""
    sc = cfg['sampling']
    K, seed = int(sc['max_zero_per_stratum']), int(sc['seed'])
    nb, S = unit_blocks(ax, unit), len(ax['segments'])
    T = np.asarray(T, np.int64)
    d, s, b, y = train_counts(ax, unit, sp, T)
    st = s * nb + b
    o = np.lexsort((d, st))
    d, st, y = d[o], st[o], y[o]
    bounds = np.searchsorted(st, np.arange(S * nb + 1))
    P, W, Y = [], [], []
    n_zero_cells = n_sampled = 0
    for k in range(S * nb):
        si, bi = divmod(k, nb)
        Tb = int(T[bi])
        pd_ = d[bounds[k]:bounds[k + 1]]
        nz = Tb - len(pd_)
        n_zero_cells += nz
        if len(pd_):
            P.append(compose(ax, unit, pd_, si, bi))
            Y.append(y[bounds[k]:bounds[k + 1]].astype(float))
            W.append(np.ones(len(pd_)))
        m = min(K, nz)
        if m > 0:
            rng = np.random.default_rng(stratum_seed(seed, unit, fold_id, ax['segments'][si], ax['blocks'][bi] if unit == '6h' else 'day'))
            zd = np.setdiff1d(np.arange(Tb, dtype=np.int64), pd_, assume_unique=True)[np.sort(rng.choice(nz, m, replace=False))]
            P.append(compose(ax, unit, zd, si, bi))
            Y.append(np.zeros(m))
            W.append(np.full(m, nz / m))
            n_sampled += m
    pos, yv, w = np.concatenate(P), np.concatenate(Y), np.concatenate(W)
    o = np.argsort(pos, kind='stable')
    pos, yv, w = pos[o], yv[o], w[o]
    full = int(S * T.sum())
    dd, _, bb = decompose(ax, unit, pos)
    summary = {'unit': unit, 'fold_id': fold_id, 'strata': S * nb, 'full_training_cells': full, 'positive_cells': int(len(d)), 'training_records': int(y.sum()),
               'zero_cells_full': int(n_zero_cells), 'zero_cells_sampled': int(n_sampled), 'sample_rows': int(len(pos)),
               'weighted_exposure': float(w.sum()), 'weighted_records': float((w * yv).sum()),
               'exposure_reconciles': bool(abs(w.sum() - full) <= 1e-6 * max(full, 1) and int(len(d)) + n_zero_cells == full),
               'records_reconcile': bool((w * yv).sum() == y.sum()), 'unique_positions': bool(len(np.unique(pos)) == len(pos)),
               'no_future_cells': bool((dd < T[bb]).all()), 'all_positives_included': bool(int((yv > 0).sum()) == len(d))}
    bad = [k for k in ('exposure_reconciles', 'records_reconcile', 'unique_positions', 'no_future_cells', 'all_positives_included') if not summary[k]]
    if bad:
        raise ExperimentError('training sample failed ' + str(bad) + ' for ' + unit + ' ' + str(fold_id))
    return pos, yv, w, summary


def features_for_positions(ctx, ax, unit, pos, batch_rows):
    """Context-variant features (the 19 builder, <= batch_rows per call) for population positions; the core variant is its leading-column slice."""
    out = []
    for a in range(0, len(pos), int(batch_rows)):
        keys = H19['keys_from_positions'](ax, unit, pos[a:a + int(batch_rows)])
        X, _ = H19['build_feature_rows'](ctx, keys, unit, 'calendar_geometry_context')
        out.append(X)
    return pd.concat(out, ignore_index=True) if out else None


In [ ]:
# FUNCTIONS (3/6): exact train-only baselines from sparse count sums and COMPLETE calendar/block denominators (never the sampled rows).
# Length-aware global/route parents, smoothed segment means (k 30/180/365), weekday/month (parent segment_mean_k30), recent-90 with fallback.
# 6 h: every count, denominator and parent is per block. Unseen segments/routes fall back explicitly (recorded); predictions are never negative.
def segment_attrs(ctx, segment_ids):
    S = ctx['seg'].loc[list(segment_ids)]
    km = S['geom_length_m'].to_numpy(float) / 1000.0
    if not (np.isfinite(km).all() and (km > 0).all()):
        raise ExperimentError('segment lengths must be finite and positive')
    return km, S['route_code'].astype(str).to_numpy(object)


def calendar_codes(dates):
    t = pd.to_datetime(pd.Series(list(dates)), format='%Y-%m-%d')
    return t.dt.dayofweek.to_numpy(np.int64), (t.dt.month - 1).to_numpy(np.int64)


def fit_baselines(ax, unit, sp, T, ctx, cfg):
    """Sufficient statistics on the training axis (segments = training universe, dates < T_b per block)."""
    bc = cfg['baselines']
    nb, S = unit_blocks(ax, unit), len(ax['segments'])
    T = np.asarray(T, np.int64)
    if (T <= 0).any():
        raise ExperimentError('a block has no training dates')
    d, s, b, y = train_counts(ax, unit, sp, T)
    km, route = segment_attrs(ctx, ax['segments'])
    wd_all, mo_all = calendar_codes(ax['dates'])
    C = np.zeros((S, nb))
    np.add.at(C, (s, b), y)
    Cwd, Cmo, Crec = np.zeros((S, 7, nb)), np.zeros((S, 12, nb)), np.zeros((S, nb))
    np.add.at(Cwd, (s, wd_all[d], b), y)
    np.add.at(Cmo, (s, mo_all[d], b), y)
    Nwd, Nmo, Nrec = np.zeros((7, nb)), np.zeros((12, nb)), np.zeros(nb)
    rd = int(bc['recent_days'])
    for bi in range(nb):
        Nwd[:, bi] = np.bincount(wd_all[:T[bi]], minlength=7)
        Nmo[:, bi] = np.bincount(mo_all[:T[bi]], minlength=12)
        lo = max(0, int(T[bi]) - rd)
        Nrec[bi] = T[bi] - lo
        m = (b == bi) & (d >= lo)
        np.add.at(Crec[:, bi], s[m], y[m])
    routes = sorted(set(route))
    ri = pd.Index(routes).get_indexer(route)
    Cr = np.zeros((len(routes), nb))
    np.add.at(Cr, ri, C)
    Lr = np.bincount(ri, weights=km, minlength=len(routes))
    Tf = T.astype(float)
    return {'unit': unit, 'nb': nb, 'segments': list(ax['segments']), 'T': Tf, 'C': C, 'Cwd': Cwd, 'Cmo': Cmo, 'Crec': Crec, 'Nwd': Nwd, 'Nmo': Nmo, 'Nrec': Nrec,
            'global_mean': C.sum(0) / (S * Tf), 'global_rate_per_km': C.sum(0) / (km.sum() * Tf),
            'routes': routes, 'route_rate_per_km': Cr / (Lr[:, None] * Tf[None, :]), 'route_counts': Cr,
            'training_records': int(y.sum()), 'training_segment_km': float(km.sum()), 'denominator_cells': int(S * T.sum())}


def baseline_tables(B, ctx, eval_segments, cfg):
    """Per-evaluation-segment prediction tables (E x [7|12 x] nb) and the explicit fallback evidence."""
    bc = cfg['baselines']
    km, route = segment_attrs(ctx, eval_segments)
    E, nb = len(km), B['nb']
    si = pd.Index(B['segments']).get_indexer(list(eval_segments))
    seen = si >= 0
    ri = pd.Index(B['routes']).get_indexer(route)
    rseen = ri >= 0
    rate = np.where(rseen[:, None], B['route_rate_per_km'][np.maximum(ri, 0)], B['global_rate_per_km'][None, :])
    P = rate * km[:, None]
    g = lambda a: np.where(seen.reshape((-1,) + (1,) * (a.ndim - 1)), a[np.maximum(si, 0)], 0.0)
    C, N = g(B['C']), np.where(seen[:, None], B['T'][None, :], 0.0)
    tabs = {'zero': np.zeros((E, nb)), 'global_mean': np.broadcast_to(B['global_mean'], (E, nb)).copy(),
            'global_length_rate': B['global_rate_per_km'][None, :] * km[:, None], 'route_length_rate': P}
    for k in bc['segment_mean_k']:
        tabs['segment_mean_k' + str(k)] = (C + k * P) / (N + k)
    M30 = (C + 30.0 * P) / (N + 30.0)
    kw, km_ = float(bc['weekday_k']), float(bc['month_k'])
    Nwd = np.where(seen[:, None, None], B['Nwd'][None], 0.0)
    Nmo = np.where(seen[:, None, None], B['Nmo'][None], 0.0)
    tabs['segment_weekday_k10'] = (g(B['Cwd']) + kw * M30[:, None, :]) / (Nwd + kw)
    tabs['segment_month_k30'] = (g(B['Cmo']) + km_ * M30[:, None, :]) / (Nmo + km_)
    use_rec = seen[:, None] & (B['Nrec'][None, :] > 0)
    tabs['recent90_segment'] = np.where(use_rec, g(B['Crec']) / np.where(B['Nrec'] > 0, B['Nrec'], 1.0)[None, :], M30)
    for k, v in tabs.items():
        if not (np.isfinite(v).all() and (v >= 0).all()):
            raise ExperimentError('baseline ' + k + ' produced a non-finite or negative mean')
    ev = {'eval_segments': E, 'unseen_segments': int((~seen).sum()), 'unseen_routes_segments': int((~rseen).sum()),
          'zero_route_rate_segments': int((rseen & (P.max(1) == 0)).sum()), 'recent_fallback_cells': int((~use_rec).sum())}
    return tabs, ev


def baseline_predict(tabs, e, b, wd, mo, names):
    out = {}
    for n in names:
        t = tabs[n]
        out[n] = t[e, wd, b] if n == 'segment_weekday_k10' else (t[e, mo, b] if n == 'segment_month_k30' else t[e, b])
    return out


In [ ]:
# FUNCTIONS (4/6): the locked estimator grid, fold-train-only preprocessing, weighted fitting with honest warning/failure records,
# the deterministic routed_context policy (feature missingness only, never labels) and hash-verified model bundles (no unverified joblib load).
def split_columns(cols, cfg):
    cats = [c for c in cols if c in cfg['candidates']['categorical_columns']]
    return cats, [c for c in cols if c not in cats]


def make_estimator(config, cols, cfg):
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import PoissonRegressor
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
    cats, nums = split_columns(cols, cfg)
    fam, p = config['family'], dict(config['params'])
    if fam in ('poisson', 'random_forest'):
        steps = [('impute', SimpleImputer(strategy='median', keep_empty_features=True))] + ([('scale', StandardScaler())] if fam == 'poisson' else [])
        pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=np.float64), cats), ('num', Pipeline(steps), nums)],
                                remainder='drop', sparse_threshold=0.0)
        model = PoissonRegressor(**p) if fam == 'poisson' else RandomForestRegressor(**p)
    elif fam == 'hist_gradient_boosting':
        pre = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan, encoded_missing_value=np.nan, dtype=np.float64), cats),
                                 ('num', 'passthrough', nums)], remainder='drop', sparse_threshold=0.0)
        model = HistGradientBoostingRegressor(categorical_features=[True] * len(cats) + [False] * len(nums), **p)
    else:
        raise ExperimentError('unknown family ' + str(fam))
    return Pipeline([('pre', pre), ('model', model)])


def fit_component(config, variant, X, y, w, cols, cfg):
    """Fit one (config, variant) on the fold training sample with sample weights and 2 threads. Returns (estimator or None, record).
    Convergence and other warnings are recorded verbatim; an exception marks the component failed (never silently replaced)."""
    from sklearn.exceptions import ConvergenceWarning
    from threadpoolctl import threadpool_limits
    rec = {'config_id': config['config_id'], 'family': config['family'], 'variant': variant, 'rows': int(len(y)), 'status': 'failed', 'error': None,
           'warnings': [], 'convergence_warning': False, 'seconds': None}
    t0 = time.time()
    est = None
    try:
        cats, _ = split_columns(cols, cfg)
        for c in cats:
            if X[c].nunique(dropna=True) > int(cfg['candidates']['max_category_cardinality']):
                raise ExperimentError('categorical ' + c + ' exceeds ' + str(cfg['candidates']['max_category_cardinality']) + ' categories')
        est = make_estimator(config, cols, cfg)
        with warnings.catch_warnings(record=True) as W:
            warnings.simplefilter('always')
            with threadpool_limits(limits=int(cfg['candidates']['threads'])):
                est.fit(X[cols], np.asarray(y, float), model__sample_weight=np.asarray(w, float))
        rec['warnings'] = [w_.category.__name__ + ': ' + str(w_.message)[:300] for w_ in W][:20]
        rec['convergence_warning'] = any(issubclass(w_.category, ConvergenceWarning) for w_ in W)
        p = predict_component(est, X.iloc[:min(len(X), 1000)], cols, cfg)
        rec['status'] = 'ok_convergence_warning' if rec['convergence_warning'] else 'ok'
        rec['train_sample_pred_mean'] = float(p.mean())
    except Exception as e:  # recorded honestly; the component (and any policy needing it) becomes ineligible
        rec['error'] = type(e).__name__ + ': ' + str(e)[:500]
        est = None
    rec['seconds'] = round(time.time() - t0, 2)
    return est, rec


def predict_component(est, X, cols, cfg):
    from threadpoolctl import threadpool_limits
    with threadpool_limits(limits=int(cfg['candidates']['threads'])):
        p = np.asarray(est.predict(X[cols]), dtype=np.float64)
    if p.shape != (len(X),) or not np.isfinite(p).all() or (p < 0).any():
        raise ExperimentError('component prediction is non-finite, negative or misshaped')
    return p


def context_complete(X, unit, cfg):
    """Routing trigger from FEATURES only: all weather (daily + block for 6 h) and the 3 AADT columns finite. bank_holiday_asof is NOT a trigger."""
    cols = cfg['candidates']['routed_trigger_columns'][unit]
    return np.isfinite(X[cols].to_numpy(float)).all(axis=1)


# Additions the routed-trigger guard must refuse (report negative controls), grouped by why they are not context predictors.
TRIGGER_REFUSED_ADDITIONS = {'label_target': ['y_count', 'y_recorded_count'], 'key_identifier': ['segment_id', 'oid', 'target_date'],
                             'event_coordinate': ['event_lat', 'event_lon'], 'bank_calendar': ['bank_holiday_asof']}


def routed_trigger_problems(cols, unit, feats, settings, cfg):
    """Exact-name routed-trigger guard (no substring heuristics); returns the list of problems, empty when valid. The trigger must be exactly
    the unit's context-only predictors = pinned context variant minus core variant, which must equal the 19 weather (daily, + block for 6 h)
    and AADT lists; every column must also pass the 19 predictor allowlist. Labels/targets, keys and event coordinates are not allowlisted;
    bank_holiday_asof and all other calendar/geometry columns are core-variant columns, so none of them can be a context trigger."""
    cols = list(cols)
    if not cols or not all(isinstance(c, str) for c in cols) or len(set(cols)) != len(cols):
        return ['trigger must be a non-empty list of unique column names: ' + str(cols)]
    core = feats[unit][cfg['candidates']['core_variant']]
    allowed = [c for c in feats[unit][cfg['candidates']['context_variant']] if c not in set(core)]
    expected = list(H19['WX_DAILY']) + (list(H19['WX_BLOCK']) if unit == '6h' else []) + list(H19['AADT'])
    probs = []
    if allowed != expected:
        probs.append('context-only predictors ' + str(allowed) + ' differ from the 19 weather/AADT lists ' + str(expected))
    try:
        H19['validate_columns'](cols, settings)
    except ValueError as e:
        probs.append('predictor allowlist: ' + str(e))
    in_core = [c for c in cols if c in set(core)]
    if in_core:
        probs.append('core-variant (calendar/bank/geometry) columns cannot trigger context: ' + str(in_core))
    other = [c for c in cols if c not in set(allowed) and c not in set(core)]
    if other:
        probs.append('not context-only predictors: ' + str(other))
    missing = [c for c in allowed if c not in set(cols)]
    if missing:
        probs.append('context-only predictors missing from the trigger: ' + str(missing))
    return probs


def routed_trigger_controls(cfg, feats, settings):
    """Report negative controls: each unit's configured trigger must pass routed_trigger_problems, and that trigger plus any refused
    label/target, key, event-coordinate or bank-calendar column must be rejected by the same guard."""
    rows = []
    for unit in UNITS:
        cols = list(cfg['candidates']['routed_trigger_columns'][unit])
        p = routed_trigger_problems(cols, unit, feats, settings, cfg)
        rows.append({'control': 'routing_trigger_is_exactly_context_weather_aadt', 'unit': unit, 'method': 'routed_trigger_problems', 'passed': not p,
                     'detail': '; '.join(p)})
        for kind, bads in TRIGGER_REFUSED_ADDITIONS.items():
            for bad in bads:
                p = routed_trigger_problems(cols + [bad], unit, feats, settings, cfg)
                rows.append({'control': 'routing_trigger_rejects_' + kind + '_' + bad, 'unit': unit, 'method': 'routed_trigger_problems', 'passed': bool(p),
                             'detail': '; '.join(p)})
    return rows


def routed_prediction(core, ctx_pred, mask):
    if core is None or ctx_pred is None:
        raise ExperimentError('routed_context needs both component predictions (component failed -> policy ineligible)')
    return np.where(mask, ctx_pred, core)


def method_names(cfg):
    """Candidate methods 'config_id::policy' (core, context, routed_context) + baselines, in fixed order."""
    out = []
    for c in cfg['candidates']['configs']:
        out += [c['config_id'] + '::core', c['config_id'] + '::context', c['config_id'] + '::routed_context']
    return out + list(cfg['baselines']['order'])


def save_bundle(est, path, meta):
    """joblib dump + trusted metadata (sha256, bytes, config, columns, lock sha); bundles are only reloaded through load_bundle."""
    import joblib
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp-' + str(os.getpid()))
    joblib.dump(est, tmp, compress=3)
    os.replace(tmp, path)
    m = dict(meta, file=path.name, sha256=sha256_file(path), bytes=path.stat().st_size)
    atomic_write_json(path.with_suffix('.meta.json'), m)
    return m


def load_bundle(path, expected_sha):
    """Refuses (no joblib.load) unless the file and its metadata sha both equal the trusted expected sha."""
    import joblib
    path = Path(path)
    meta = read_strict_json(path.with_suffix('.meta.json'))
    if not expected_sha or meta.get('sha256') != expected_sha:
        raise ExperimentError('model bundle metadata sha differs from the trusted record: ' + str(path))
    verify_pinned(path, expected_sha, 'model bundle')
    return joblib.load(path), meta


In [ ]:
# FUNCTIONS (5/6): streamed metric accumulators (exactly mergeable across batches/folds/strata), Poisson deviance with the floor ONLY inside the
# metric, deterministic selection gate, and the paired 7-calendar-day block bootstrap.
ACC = ('n', 'dev', 'abs', 'sq', 'pred', 'y', 'n_zero', 'n_nonzero', 'dev_zero', 'dev_nonzero', 'pred_zero', 'y_nonzero', 'pred_nonzero')


def poisson_deviance_terms(y, mu, floor):
    """Unit deviance 2*(y*log(y/m) - (y - m)), m = max(mu, floor) (floor used ONLY here); y*log(y/m) := 0 for y == 0. y must be >= 0."""
    y, mu = np.asarray(y, float), np.asarray(mu, float)
    if (y < 0).any() or not np.isfinite(mu).all() or (mu < 0).any():
        raise ExperimentError('deviance needs y >= 0 and finite non-negative predictions')
    m = np.maximum(mu, float(floor))
    ylog = np.where(y > 0, y * np.log(np.where(y > 0, y, 1.0) / m), 0.0)
    return 2.0 * (ylog - (y - m))


def acc_new(k=1):
    return {f: np.zeros(k) for f in ACC}


def acc_add(acc, y, mu, floor, codes=None, k=1):
    """Add rows to acc (k groups; codes = group per row, None = single group)."""
    return acc_add_dv(acc, y, mu, poisson_deviance_terms(y, mu, floor), codes, k)


def acc_add_dv(acc, y, mu, dv, codes=None, k=1):
    y, mu = np.asarray(y, float), np.asarray(mu, float)
    c = np.zeros(len(y), np.int64) if codes is None else np.asarray(codes, np.int64)
    z = y == 0
    for f, v in (('n', np.ones(len(y))), ('dev', dv), ('abs', np.abs(mu - y)), ('sq', (mu - y) ** 2), ('pred', mu), ('y', y), ('n_zero', z * 1.0),
                 ('n_nonzero', (~z) * 1.0), ('dev_zero', dv * z), ('dev_nonzero', dv * ~z), ('pred_zero', mu * z), ('y_nonzero', y * ~z), ('pred_nonzero', mu * ~z)):
        acc[f] += np.bincount(c, weights=v, minlength=k)
    return acc


def acc_merge(*accs):
    out = {f: np.zeros_like(accs[0][f]) for f in ACC}
    for a in accs:
        for f in ACC:
            out[f] = out[f] + a[f]
    return out


def acc_metrics(acc, i=0):
    g = {f: float(acc[f][i]) for f in ACC}
    n = g['n']
    if n <= 0:
        return {'n': 0}
    return {'n': int(n), 'mean_poisson_deviance': g['dev'] / n, 'mae': g['abs'] / n, 'rmse': math.sqrt(g['sq'] / n), 'bias': (g['pred'] - g['y']) / n,
            'wape': g['abs'] / g['y'] if g['y'] > 0 else None, 'total_pred': g['pred'], 'total_actual': g['y'],
            'pred_over_actual': g['pred'] / g['y'] if g['y'] > 0 else None, 'zero_cells': int(g['n_zero']), 'nonzero_cells': int(g['n_nonzero']),
            'mean_deviance_zero_cells': g['dev_zero'] / g['n_zero'] if g['n_zero'] else None,
            'mean_deviance_nonzero_cells': g['dev_nonzero'] / g['n_nonzero'] if g['n_nonzero'] else None,
            'mean_pred_zero_cells': g['pred_zero'] / g['n_zero'] if g['n_zero'] else None,
            'mean_pred_nonzero_cells': g['pred_nonzero'] / g['n_nonzero'] if g['n_nonzero'] else None}


def strongest_baseline(pooled, cfg):
    names = [b for b in cfg['baselines']['order'] if b not in cfg['baselines']['ineligible'] and b in pooled]
    if not names:
        raise ExperimentError('no eligible nonzero baseline was evaluated')
    return min(names, key=lambda b: (pooled[b]['mean_poisson_deviance'], names.index(b)))


def select_unit(pooled, per_fold, status, cfg):
    """pooled: method -> metrics; per_fold: fold_id -> method -> metrics; status: method -> True when every fold's required components fitted.
    Best deployment-eligible candidate (core / routed_context; ties core, then config order) vs the strongest nonzero baseline; gates from the lock."""
    sel, folds = cfg['selection'], [f['fold_id'] for f in cfg['time']['folds']]
    pol = cfg['candidates']['eligible_policies']
    cids = [c['config_id'] for c in cfg['candidates']['configs']]
    fam = {c['config_id']: c['family'] for c in cfg['candidates']['configs']}
    cands = [(cid + '::' + p, cid, p) for cid in cids for p in pol if status.get(cid + '::' + p) and cid + '::' + p in pooled]
    base = strongest_baseline(pooled, cfg)
    key = lambda t: (pooled[t[0]]['mean_poisson_deviance'], pol.index(t[2]), cids.index(t[1]))
    per_family = {}
    for t in sorted(cands, key=key):
        per_family.setdefault(fam[t[1]], t[0])
    out = {'strongest_baseline': base, 'baseline_pooled': pooled[base], 'eligible_candidates': [t[0] for t in cands], 'best_per_family': per_family,
           'ineligible_methods': sorted(m for m, ok in status.items() if not ok)}
    if not cands:
        return dict(out, best_candidate=None, recommended='baseline', selected_method=base, reason='no deployment-eligible candidate fitted in every fold')
    best = min(cands, key=key)[0]
    db, dc = pooled[base]['mean_poisson_deviance'], pooled[best]['mean_poisson_deviance']
    mb, mc = pooled[base]['mae'], pooled[best]['mae']
    red = (db - dc) / db if db > 0 else None
    ratio = mc / mb if mb > 0 else (1.0 if mc == 0 else None)
    wins = {f: bool(per_fold[f][best]['mean_poisson_deviance'] < per_fold[f][base]['mean_poisson_deviance']) for f in folds}
    gates = {'deviance_reduction': red is not None and red >= float(sel['min_relative_deviance_reduction']),
             'mae_ratio': ratio is not None and ratio <= float(sel['max_mae_ratio']), 'fold_wins': sum(wins.values()) >= int(sel['min_fold_wins'])}
    ok = all(gates.values())
    return dict(out, best_candidate=best, candidate_pooled=pooled[best], relative_deviance_reduction=red, mae_ratio=ratio, fold_wins=wins,
                n_fold_wins=sum(wins.values()), gates=gates, recommended='candidate' if ok else 'baseline', selected_method=best if ok else base,
                reason='all gates passed' if ok else 'gate(s) failed: ' + ', '.join(k for k, v in gates.items() if not v))


def day_blocks(n_days, block_days):
    """Consecutive non-overlapping calendar blocks from the first day; the final partial block (n_days % block_days days) is its own block."""
    return [(a, min(a + int(block_days), n_days)) for a in range(0, n_days, int(block_days))]


def block_bootstrap(dev_c_day, dev_b_day, cfg):
    """Paired bootstrap of 1 - sum dev(candidate)/sum dev(baseline) resampling whole 7-day calendar blocks (every segment/block of a day moves
    together). Returns point estimate and percentile CI, or no CI (with the reason) when any resample is undefined."""
    bs = cfg['bootstrap']
    c, b = np.asarray(dev_c_day, float), np.asarray(dev_b_day, float)
    if c.shape != b.shape or c.ndim != 1 or len(c) == 0:
        raise ExperimentError('daily contribution arrays must be aligned 1-D')
    bl = day_blocks(len(c), bs['block_days'])
    sc, sb = np.asarray([c[i:j].sum() for i, j in bl]), np.asarray([b[i:j].sum() for i, j in bl])
    rng = np.random.default_rng(int(bs['seed']))
    idx = rng.integers(0, len(bl), size=(int(bs['resamples']), len(bl)))
    Rc, Rb = sc[idx].sum(1), sb[idx].sum(1)
    valid = Rb > 0
    point = 1.0 - c.sum() / b.sum() if b.sum() > 0 else None
    res = {'days': len(c), 'blocks': len(bl), 'block_days': int(bs['block_days']), 'partial_last_block_days': len(c) % int(bs['block_days']),
           'resamples': int(bs['resamples']), 'valid_resamples': int(valid.sum()), 'seed': int(bs['seed']), 'point_relative_deviance_reduction': point,
           'ci_low': None, 'ci_high': None, 'ci_status': None}
    if point is None or not valid.all():
        res['ci_status'] = 'undefined: baseline deviance sum is zero in ' + str(int((~valid).sum())) + ' resample(s) or overall'
        return res
    stat = 1.0 - Rc / Rb
    lo, hi = np.quantile(stat, bs['ci'])
    return dict(res, ci_low=float(lo), ci_high=float(hi), ci_status='ok (exploratory, post-selection, not confirmatory)')


In [ ]:
# FUNCTIONS (6/6): full-population streamed evaluation (features built ONCE per batch with the 19 builder and reused by every component,
# policy and baseline), float64 .npy memmap artifacts with NaN sentinel + completeness checks, artifact budget, geometry-only transfer split,
# figures, Thai narrative from computed results, and the strict manifest.
PRED_BIN_EDGES = [0.0, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 0.1, 0.3, 1.0]
CAVEATS = ['Target = SELECTED+LINKED recorded database counts (status_50m matched AND segment assigned), NOT all physical accidents; zero = no such record.',
           'Predictions are expected recorded counts per segment x day / 6 h block, not per-vehicle or personal probabilities.',
           '2024/25 are PREVIOUSLY SEEN diagnostics (not an untouched test); 30-day label latency is a research assumption.',
           'Current road snapshot applied retrospectively; coarse province-point weather and survey-point AADT proxies; reporting/linkage selection bias remains.',
           'Spatial transfer is a contiguous unseen-SECTION diagnostic (northeast block), not all-new-route or untouched-future performance.']


def safe_name(method):
    return method.replace('::', '__')


def new_array(path, n, dtype=np.float64):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    mm = np.lib.format.open_memmap(path, mode='w+', dtype=dtype, shape=(int(n),))
    mm[:] = np.nan if np.dtype(dtype).kind == 'f' else 255  # sentinels: every row must be written
    return mm


def finalize_array(path, n, kind):
    """Reload with allow_pickle=False; exact dtype/shape; preds/labels finite and >= 0 (no NaN sentinel left); masks in {0, 1}."""
    a = np.load(path, mmap_mode='r', allow_pickle=False)
    want = np.uint8 if kind == 'mask' else np.float64
    if a.dtype != want or a.shape != (int(n),):
        raise ExperimentError('array ' + str(path) + ' has dtype/shape ' + str(a.dtype) + str(a.shape))
    for i in range(0, int(n), 1 << 21):
        c = np.asarray(a[i:i + (1 << 21)])
        if kind == 'mask' and not np.isin(c, (0, 1)).all():
            raise ExperimentError('mask ' + str(path) + ' incomplete')
        if kind != 'mask' and (not np.isfinite(c).all() or (c < 0).any()):
            raise ExperimentError('array ' + str(path) + ' incomplete, non-finite or negative')
    return {'file': Path(path).name, 'kind': kind, 'dtype': str(a.dtype), 'shape': [int(n)], 'sha256': sha256_file(path), 'bytes': Path(path).stat().st_size}


def estimate_artifact_bytes(cfg, rows):
    """rows: {'oof': {unit: n}, 'diag': {unit: n}, 'transfer': {unit: n}}. Fails (before any fit) when the 4 GiB budget would be exceeded."""
    bpv, nc, nbase = 8, len(cfg['candidates']['configs']), len(cfg['baselines']['order']) - 1  # the all-zero baseline is never stored
    parts = {}
    for u, n in rows['oof'].items():
        parts['oof_' + u] = n * (bpv * (2 * nc + nbase + 1) + 1)
    for u, n in rows['diag'].items():
        parts['diag_' + u] = n * (bpv * 3 + 1)
    for u, n in rows['transfer'].items():
        parts['transfer_' + u] = n * bpv * (len(cfg['transfer']['configs']) + nbase + 1)
    parts['models_allowance'] = 512 * 1024 ** 2
    tot = int(sum(parts.values()))
    res = {'parts': parts, 'total_bytes': tot, 'budget_bytes': int(cfg['resources']['artifact_budget_bytes']), 'within_budget': tot <= int(cfg['resources']['artifact_budget_bytes'])}
    if not res['within_budget']:
        raise ExperimentError('estimated artifacts ' + str(tot) + ' B exceed the locked budget; report before any protocol change (evaluation is never sampled)')
    return res


def row_strata(X, y, b, mask, mu_bins_for=None):
    fest = [c for c in X.columns if c.startswith('fest_') and not c.endswith('_dist')]
    season = np.argmax(X[['season_hot', 'season_wet', 'season_cool']].to_numpy(float), axis=1)
    return {'time_block': b, 'season': season, 'festival_window': (X[fest].to_numpy(float) > 0).any(axis=1).astype(np.int64),
            'context_complete': mask.astype(np.int64), 'actual_zero': (np.asarray(y) == 0).astype(np.int64)}


STRATA_LABELS = {'season': ['hot', 'wet', 'cool'], 'festival_window': ['no', 'yes'], 'context_complete': ['no', 'yes'], 'actual_zero': ['nonzero', 'zero']}


def evaluate_population(ctx, ax, unit, sp, periods, models, tabs, cfg, arr_dir, prefix, save_methods, save_mask=True):
    """Stream EVERY cell of ax over the contiguous periods [(pid, start, end)] in <= batch_rows batches. Per batch: one 19 feature build
    (context variant; core = column slice), every fitted component, routed_context, and every baseline. Saves float64 arrays only for
    save_methods (+ y and the routing mask) in implicit date->segment->block order from the first period start. Returns accumulators."""
    check_sparse(sp, ax, unit)
    floor, br = float(cfg['metrics']['deviance_floor']), int(cfg['resources']['batch_rows'])
    nb, S = unit_blocks(ax, unit), len(ax['segments'])
    per_day = S * nb
    d0 = ax['date_pos'][periods[0][1]]
    n_days = ax['date_pos'][periods[-1][2]] - d0 + 1
    n = n_days * per_day
    core_cols = H19['feature_columns'](ctx['settings'], unit, 'calendar_geometry')
    ctx_cols = H19['feature_columns'](ctx['settings'], unit, 'calendar_geometry_context')
    wd_all, mo_all = calendar_codes(ax['dates'])
    cids = [c['config_id'] for c in cfg['candidates']['configs']]
    bnames = list(cfg['baselines']['order'])
    arr_dir = Path(arr_dir)
    files = {m: new_array(arr_dir / (prefix + '_' + unit + '_' + safe_name(m) + '.npy'), n) for m in save_methods}
    yarr = new_array(arr_dir / (prefix + '_' + unit + '_y.npy'), n)
    marr = new_array(arr_dir / (prefix + '_' + unit + '_context_mask.npy'), n, np.uint8) if save_mask else None
    ks = {'time_block': nb, 'season': 3, 'festival_window': 2, 'context_complete': 2, 'actual_zero': 2, 'pred_bin': len(PRED_BIN_EDGES)}
    period_acc, strata, day_dev, ctx_share, written = {}, {}, {}, {}, 0
    t0 = time.time()
    for pid, start, end in periods:
        pa = period_acc.setdefault(pid, {})
        cc = [0, 0]
        for a, keys in H19['iter_population_batches'](ax, unit, br, start, end):
            pos = np.arange(a, a + len(keys), dtype=np.int64)
            loc = pos - d0 * per_day
            y = H19['lookup_counts'](sp, pos).astype(np.float64)
            X, _ = H19['build_feature_rows'](ctx, keys, unit, 'calendar_geometry_context')
            if list(X.columns) != ctx_cols:
                raise ExperimentError('feature columns changed during evaluation')
            mask = context_complete(X, unit, cfg)
            cc[0] += int(mask.sum())
            cc[1] += len(mask)
            d, s, b = decompose(ax, unit, pos)
            P = {}
            for cid in cids:
                mc, mx = models[pid]['core'].get(cid), models[pid]['context'].get(cid)
                pc = predict_component(mc, X, core_cols, cfg) if mc is not None else None
                px = predict_component(mx, X, ctx_cols, cfg) if mx is not None else None
                if pc is not None:
                    P[cid + '::core'] = pc
                if px is not None:
                    P[cid + '::context'] = px
                if pc is not None and px is not None:
                    P[cid + '::routed_context'] = routed_prediction(pc, px, mask)
            P.update(baseline_predict(tabs[pid], s, b, wd_all[d], mo_all[d], bnames))
            st = row_strata(X, y, b, mask)
            yarr[loc] = y
            if marr is not None:
                marr[loc] = mask.astype(np.uint8)
            for m, mu in P.items():
                dv = poisson_deviance_terms(y, mu, floor)
                acc_add_dv(pa.setdefault(m, acc_new()), y, mu, dv)
                sm = strata.setdefault(m, {k: acc_new(v) for k, v in ks.items()})
                for k, codes in st.items():
                    acc_add_dv(sm[k], y, mu, dv, codes, ks[k])
                acc_add_dv(sm['pred_bin'], y, mu, dv, np.searchsorted(PRED_BIN_EDGES, mu, side='right') - 1, ks['pred_bin'])
                day_dev.setdefault(m, np.zeros(n_days))
                day_dev[m] += np.bincount(d - d0, weights=dv, minlength=n_days)
                if m in files:
                    files[m][loc] = mu
            missing = [m for m in save_methods if m not in P]
            if missing:
                raise ExperimentError('cannot save methods without predictions: ' + str(missing))
            written += len(pos)
        ctx_share[pid] = {'rows': cc[1], 'context_complete_rows': cc[0], 'share': cc[0] / cc[1] if cc[1] else None}
    if written != n:
        raise ExperimentError('evaluation wrote ' + str(written) + ' rows, population has ' + str(n) + ' (never downsampled)')
    meta = {}
    for m, mm in list(files.items()) + [('y', yarr)] + ([('context_mask', marr)] if marr is not None else []):
        mm.flush()
        p = Path(mm.filename)
        del mm
        meta[m] = finalize_array(p, n, 'mask' if m == 'context_mask' else ('y' if m == 'y' else 'pred'))
    files.clear()
    return {'unit': unit, 'rows': n, 'first_date': periods[0][1], 'last_date': periods[-1][2], 'segments': S, 'blocks': nb,
            'order': 'date, then segment_id (sorted), then time_block', 'periods': [{'id': p, 'start': s0, 'end': s1, 'row_offset': (ax['date_pos'][s0] - d0) * per_day,
            'rows': (ax['date_pos'][s1] - ax['date_pos'][s0] + 1) * per_day} for p, s0, s1 in periods],
            'arrays': meta, 'period_acc': period_acc, 'strata': strata, 'day_dev': day_dev, 'context_share': ctx_share, 'seconds': round(time.time() - t0, 1)}


def metric_rows(accs, **extra):
    return [dict(extra, method=m, **acc_metrics(a)) for m, a in accs.items()]


def strata_rows(strata, unit, scope):
    rows = []
    for m, sm in strata.items():
        for k, acc in sm.items():
            for i in range(len(acc['n'])):
                if acc['n'][i] > 0:
                    lab = STRATA_LABELS[k][i] if k in STRATA_LABELS else (str(PRED_BIN_EDGES[i]) if k == 'pred_bin' else str(i))
                    rows.append(dict(unit=unit, scope=scope, method=m, stratum=k, level=lab, **acc_metrics(acc, i)))
    return rows


# ---------- geometry-only spatial transfer split ----------
def ids_sha(ids, encoding):
    s = sorted(ids)
    if encoding == 'lines':
        return sha256_bytes('\n'.join(s).encode('utf-8'))
    if encoding == 'lines_trailing_newline':
        return sha256_bytes(('\n'.join(s) + '\n').encode('utf-8'))
    if encoding == 'json_list':
        return sha256_bytes(json.dumps(s).encode('utf-8'))
    raise ExperimentError('unknown id hash encoding ' + str(encoding))


def transfer_split(seg, geojson_path, prereg, encodings):
    """Recompute the manager split from accepted 19 GEOMETRY only (midpoints, source OIDs, EPSG:32647 lines; no labels/scores/event coords):
    eval = midpoint lon >= L AND lat >= A; training excludes eval-source-OID segments and segments within the buffer of any eval segment."""
    from pyproj import Transformer
    from shapely import STRtree
    from shapely.geometry import LineString
    pr = prereg['eval_midpoint_predicate']
    ids = seg.index.to_numpy(object)
    ev = (seg['geom_mid_lon'].to_numpy(float) >= float(pr['lon_gte'])) & (seg['geom_mid_lat'].to_numpy(float) >= float(pr['lat_gte']))
    oid = seg['oid'].to_numpy(float)
    if not (np.isfinite(oid).all() and (oid == np.round(oid)).all()):
        raise ExperimentError('universe source OIDs must be integral')
    eoids = set(oid[ev].astype(np.int64).tolist())
    same = ~ev & np.isin(oid.astype(np.int64), list(eoids))
    fc = json.loads(Path(geojson_path).read_text('utf-8'))
    fwd = Transformer.from_crs(4326, 32647, always_xy=True)
    geom = {}
    for f in fc['features']:
        c = np.asarray(f['geometry']['coordinates'], float)
        x, y_ = fwd.transform(c[:, 0], c[:, 1])
        geom[f['id']] = LineString(np.column_stack([x, y_]))
    if set(geom) != set(ids):
        raise ExperimentError('universe geometry ids differ from the universe segments')
    eg = [geom[i] for i in ids[ev]]
    tree = STRtree(eg)
    buf = float(prereg['buffer_m_epsg32647'])
    dist = np.full(len(ids), np.inf)
    for j in np.flatnonzero(~ev & ~same):
        g = geom[ids[j]]
        near = tree.query(g, predicate='dwithin', distance=buf)
        dist[j] = min((g.distance(eg[k]) for k in near), default=np.inf)
    drop = ~ev & ~same & (dist < buf)
    tr = ~ev & ~same & ~drop
    eval_ids, train_ids = sorted(ids[ev].tolist()), sorted(ids[tr].tolist())
    min_d = min((float(np.min(tree.query_nearest(geom[i], return_distance=True)[1])) for i in train_ids), default=np.inf)  # independent of the drop rule
    match = {e: (ids_sha(eval_ids, e) == prereg['eval_segment_ids_sha256'] and ids_sha(train_ids, e) == prereg['train_segment_ids_sha256']) for e in encodings}
    toids = set(oid[tr].astype(np.int64).tolist())
    res = {'eval_segments': len(eval_ids), 'eval_source_oids': len(eoids), 'train_segments': len(train_ids), 'train_source_oids': len(toids),
           'segments_sharing_eval_source_oid': int(ev.sum() + same.sum()), 'removed_same_source_oid_non_eval': int(same.sum()), 'removed_buffer_additional': int(drop.sum()),
           'source_oids_disjoint': not (eoids & toids), 'min_train_eval_distance_m': None if not np.isfinite(min_d) else float(min_d),
           'buffer_respected': bool(not np.isfinite(min_d) or min_d >= buf), 'id_hash_match_by_encoding': match,
           'matched_encoding': next((e for e, ok in match.items() if ok), None)}
    checks = {'eval_count': res['eval_segments'] == prereg['eval_segments'], 'train_count': res['train_segments'] == prereg['train_segments'],
              'eval_oids': res['eval_source_oids'] == prereg['eval_source_oids'], 'train_oids': res['train_source_oids'] == prereg['train_source_oids'],
              'removed_buffer': res['removed_buffer_additional'] == prereg['removed_buffer_additional'], 'disjoint': res['source_oids_disjoint'],
              'buffer': res['buffer_respected'], 'ids_sha': res['matched_encoding'] is not None}
    res['checks'] = checks
    if not all(checks.values()):
        raise ExperimentError('spatial transfer split does not reproduce the manager preregistration: ' + str([k for k, v in checks.items() if not v]))
    return eval_ids, train_ids, res


# ---------- figures, narrative, manifest ----------
def set_fonts(cfg):
    import matplotlib
    from matplotlib import font_manager
    have = {f.name for f in font_manager.fontManager.ttflist}
    use = [f for f in cfg['figure_fonts'] if f in have]
    matplotlib.rcParams['font.family'] = use + ['DejaVu Sans'] if use else ['DejaVu Sans']
    return use


def save_fig(fig, path):
    import matplotlib.pyplot as plt
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=130, bbox_inches='tight')
    plt.close(fig)
    if path.stat().st_size < 5000:
        raise ExperimentError('figure ' + path.name + ' looks empty')
    return path.name


def make_figures(fig_dir, LB, FM, CAL, TR, SEL):
    """Actual-result figures: pooled OOF deviance per method, fold deviance of the selected-best candidate vs strongest baseline,
    calibration by predicted-value bin, and the transfer diagnostic. Inputs are the computed tables only."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    out = []
    fig, axs = plt.subplots(1, 2, figsize=(15, 7))
    for ax_, u in zip(axs, UNITS):
        d = LB[LB['unit'] == u].sort_values('mean_poisson_deviance')
        col = ['tab:orange' if m in (SEL[u]['strongest_baseline'],) else ('tab:green' if m == SEL[u].get('best_candidate') else 'tab:blue') for m in d['method']]
        ax_.barh(d['method'], d['mean_poisson_deviance'], color=col)
        ax_.invert_yaxis()
        ax_.set_title(u + ': pooled 2022-23 OOF mean Poisson deviance (recorded counts)')
        ax_.set_xlabel('mean Poisson deviance (lower is better)')
        ax_.tick_params(axis='y', labelsize=7)
    out.append(save_fig(fig, Path(fig_dir) / 'oof_deviance_by_method.png'))
    fig, axs = plt.subplots(1, 2, figsize=(13, 5))
    for ax_, u in zip(axs, UNITS):
        for m, c in ((SEL[u].get('best_candidate'), 'tab:green'), (SEL[u]['strongest_baseline'], 'tab:orange')):
            if m:
                d = FM[(FM['unit'] == u) & (FM['method'] == m)]
                ax_.plot(d['fold_id'], d['mean_poisson_deviance'], marker='o', color=c, label=m)
        ax_.set_title(u + ': fold deviance, best candidate vs strongest baseline')
        ax_.legend(fontsize=8)
    out.append(save_fig(fig, Path(fig_dir) / 'fold_deviance_candidate_vs_baseline.png'))
    fig, axs = plt.subplots(1, 2, figsize=(13, 6))
    for ax_, u in zip(axs, UNITS):
        for m, c in ((SEL[u].get('best_candidate'), 'tab:green'), (SEL[u]['strongest_baseline'], 'tab:orange')):
            if m:
                d = CAL[(CAL['unit'] == u) & (CAL['method'] == m) & (CAL['n'] > 0)]
                ax_.loglog(d['total_pred'] / d['n'], np.maximum(d['total_actual'] / d['n'], 1e-6), marker='o', color=c, label=m)
        lim = [1e-5, 1]
        ax_.plot(lim, lim, 'k--', lw=0.8)
        ax_.set_xlabel('mean predicted recorded count (bin)')
        ax_.set_ylabel('mean actual recorded count (floored 1e-6 for display)')
        ax_.set_title(u + ': OOF calibration by predicted bin')
        ax_.legend(fontsize=8)
    out.append(save_fig(fig, Path(fig_dir) / 'oof_calibration.png'))
    if TR is not None and len(TR):
        fig, axs = plt.subplots(1, 2, figsize=(14, 6))
        for ax_, u in zip(axs, UNITS):
            d = TR[TR['unit'] == u].sort_values('mean_poisson_deviance')
            ax_.barh(d['method'], d['mean_poisson_deviance'], color='tab:purple')
            ax_.invert_yaxis()
            ax_.set_title(u + ': 2023 held-out northeast sections (transfer diagnostic)')
            ax_.tick_params(axis='y', labelsize=7)
        out.append(save_fig(fig, Path(fig_dir) / 'spatial_transfer_deviance.png'))
    return out


def fmt(v, nd=4):
    return 'n/a' if v is None or (isinstance(v, float) and not math.isfinite(v)) else (('{:.' + str(nd) + 'g}').format(v) if isinstance(v, float) else str(v))


def narrative_th(SEL, BOOT, DIAG, TRS, lock_sha):
    """Thai results narrative generated ONLY from computed result objects (no hand-written score)."""
    L = ['# ผลการทดลองแบบจำลองจำนวนบันทึกอุบัติเหตุเชิงพื้นที่ (V3, notebook 20)', '',
         '**ความหมายของเป้าหมาย:** จำนวนบันทึกในฐานข้อมูลที่ผ่านการคัดเลือกและเชื่อมโยงกับช่วงถนน (SELECTED+LINKED) ต่อช่วงถนน x วัน (หรือ x ช่วง 6 ชั่วโมง) '
         'ไม่ใช่จำนวนอุบัติเหตุจริงทั้งหมด และไม่ใช่ความน่าจะเป็นต่อยานพาหนะหรือต่อบุคคล', '', 'Experiment lock sha256: `' + lock_sha + '`', '']
    for u in UNITS:
        s = SEL[u]
        L += ['## หน่วย ' + u, '',
              '- baseline ที่แข็งแรงที่สุด (OOF 2022-23 ทั้งประชากร): `' + s['strongest_baseline'] + '` deviance เฉลี่ย ' + fmt(s['baseline_pooled']['mean_poisson_deviance']),
              '- ผู้สมัครที่ดีที่สุด (นโยบายที่ใช้งานได้จริง core/routed_context): `' + str(s.get('best_candidate')) + '`' +
              ((' deviance เฉลี่ย ' + fmt(s['candidate_pooled']['mean_poisson_deviance'])) if s.get('candidate_pooled') else ''),
              '- การลดลงของ deviance สัมพัทธ์: ' + fmt(s.get('relative_deviance_reduction')) + ', อัตราส่วน MAE: ' + fmt(s.get('mae_ratio')) +
              ', ชนะใน ' + fmt(s.get('n_fold_wins')) + '/4 fold',
              '- ผลการคัดเลือก (ล็อกก่อนดูผล 2024/25): **' + s['recommended'] + '** -> `' + s['selected_method'] + '` (' + s['reason'] + ')']
        b = BOOT.get(u)
        if b:
            L.append('- bootstrap แบบบล็อก 7 วัน (สำรวจ ไม่ใช่การยืนยัน): ค่าประมาณ ' + fmt(b['point_relative_deviance_reduction']) + ', ช่วง 95% ' +
                     (fmt(b['ci_low']) + ' ถึง ' + fmt(b['ci_high']) if b['ci_low'] is not None else 'ไม่มี (' + str(b['ci_status']) + ')'))
        for r in DIAG.get(u, []):
            L.append('- ปี ' + r['year'] + ' (เคยเห็นแล้ว ไม่ใช่ชุดทดสอบที่ไม่เคยแตะ): `' + r['method'] + '` deviance ' + fmt(r['mean_poisson_deviance']) + ', MAE ' + fmt(r['mae']))
        for r in TRS.get(u, []):
            L.append('- การถ่ายโอนเชิงพื้นที่ 2023 (ช่วงถนนตะวันออกเฉียงเหนือที่กันไว้): `' + r['method'] + '` deviance ' + fmt(r['mean_poisson_deviance']))
        L.append('')
    L += ['## ข้อจำกัด', ''] + ['- ' + c for c in CAVEATS]
    return NL.join(L) + NL


def build_manifest(status, started, cfg, cfg_sha, lock_sha, nb_path, checks, report_files, data_files, protected_before, protected_after, extra):
    changed = sorted(k for k in set(protected_before) | set(protected_after) if protected_before.get(k) != protected_after.get(k))
    return {'status': status, 'version': cfg['version'], 'purpose': 'spatial recorded-count research experiments; NOT serving, colours or a production model',
            'label_meaning': cfg['label_meaning'], 'started_utc': started, 'finished_utc': utcnow(), 'config_sha256': cfg_sha, 'experiment_lock_sha256': lock_sha,
            'notebook': cfg['notebook'], 'environment': environment(), 'checks': checks, 'failed_checks': [c['check'] for c in checks if c['status'] != 'pass'],
            'report_files': report_files, 'data_files_gitignored': data_files, 'protected_files_hashed': len(protected_before), 'protected_unchanged': not changed,
            'protected_changed': changed, 'protected_sha256_before': protected_before,
            'readiness': {'production_ready': False, 'serving_ready': False, 'caveats': CAVEATS}, **extra}


In [ ]:
# SETUP: config, dependency check, output guard BEFORE any write, protected hashes, check/stage helpers.
import contextlib
import resource

STARTED = utcnow()
cfg = json.loads(CONFIG_PATH.read_text('utf-8'))
CFG_SHA = sha256_file(CONFIG_PATH)
require_modules(['pandas', 'numpy', 'sklearn', 'joblib', 'threadpoolctl', 'shapely', 'pyproj', 'matplotlib'], cfg['dependency_pins'])
PATHS = output_paths(OUTPUT_ROOT, cfg)
check_outputs(PATHS, BACKEND, cfg)
NB_PATH = BACKEND / cfg['notebook']
PROTECTED_BEFORE = hash_paths(BACKEND, protected_rels(cfg))
CHECKS, T, REPORT_FILES, DURATIONS, NEG = [], {}, {}, {}, []
BR = int(cfg['resources']['batch_rows'])
CORE_V, CTX_V = cfg['candidates']['core_variant'], cfg['candidates']['context_variant']
CONFIGS = cfg['candidates']['configs']
CIDS = [c['config_id'] for c in CONFIGS]
FOLD_IDS = [f['fold_id'] for f in cfg['time']['folds']]
NONZERO_BASE = [b for b in cfg['baselines']['order'] if b not in cfg['baselines']['ineligible']]
for k in ('report_dir', 'figure_dir', 'data_dir', 'array_dir', 'model_dir'):
    PATHS[k].mkdir(parents=True, exist_ok=False if k in ('report_dir', 'data_dir') else True)


def check(name, ok, evidence=None):
    CHECKS.append({'check': name, 'status': 'pass' if ok else 'fail', 'evidence': json.dumps(jsonable(evidence), ensure_ascii=False, allow_nan=False)[:2000]})
    if not ok:
        raise ExperimentError('check failed: ' + name + ' ' + str(evidence)[:500])


def peak_rss_bytes():
    r = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return int(r if platform.system() == 'Darwin' else r * 1024)


@contextlib.contextmanager
def stage(name):
    t0 = time.time()
    print('[stage]', name, flush=True)
    yield
    DURATIONS[name] = {'seconds': round(time.time() - t0, 2), 'peak_rss_bytes_so_far': peak_rss_bytes()}
    print('  done', name, DURATIONS[name], flush=True)


def put_report(name, obj):
    check_report_file(name, cfg)
    p = PATHS['report_dir'] / name
    if isinstance(obj, pd.DataFrame):
        atomic_write_bytes(p, csv_bytes(obj.assign(lock_sha256=LOCK_SHA)))
    elif isinstance(obj, (bytes, bytearray)):
        atomic_write_bytes(p, bytes(obj))
    else:
        atomic_write_json(p, obj)
    REPORT_FILES[name] = sha256_file(p)


def put_data(name, obj):
    p = PATHS['data_dir'] / name
    if isinstance(obj, pd.DataFrame):
        atomic_write_bytes(p, csv_bytes(obj.assign(lock_sha256=LOCK_SHA)))
    else:
        atomic_write_json(p, obj)
    return sha256_file(p)


In [ ]:
# INPUTS + LOCK: manager pins, every input/prepared-data hash, verified 19 FUNCTIONS, prepared context via 19, sparse targets with the 19 bound
# contract, fold/embargo table, artifact budget, builder parity -- then the immutable experiment lock is persisted BEFORE any fit or score.
with stage('inputs_and_lock'):
    PINS, PINS_SHA = load_pins(BACKEND, cfg)
    INPUT_ROWS, DOCS = verify_inputs(BACKEND, cfg, PINS)
    check('every pinned input and all prepared data files hash-verified; 19 manifest complete and consistent with the manager pins', True, {'inputs': len(INPUT_ROWS)})
    _, NB19_SHA, NB19_CELLS = load_nb19_functions(BACKEND / cfg['inputs']['notebook19']['path'], PINS, cfg['inputs']['notebook19']['functions_cells'])
    check('notebook-19 FUNCTIONS and code-cell source hashes equal the manager pins', NB19_SHA == PINS['notebook19_functions_sha256'], {'functions_sha256': NB19_SHA})
    DD19 = BACKEND / cfg['inputs']['prepared_data_dir']
    CTX = H19['load_prepared_context'](DD19, DOCS['preparation_manifest']['data_files_gitignored'])
    FEATS = check_feature_columns(CTX['settings'], PINS)
    TRIG = {u: routed_trigger_problems(cfg['candidates']['routed_trigger_columns'][u], u, FEATS, CTX['settings'], cfg) for u in UNITS}
    check('routed trigger columns are exactly the context-only weather/AADT predictors (no label, key, event or bank-calendar column)', not any(TRIG.values()), TRIG)
    AXJ = read_strict_json(DD19 / 'population_axes.json')
    AX = H19['population_axes'](AXJ['segments'], AXJ['dates'][0], AXJ['dates'][-1], AXJ['blocks'])
    check('population axes reproduce the 19 axes file and universe', AX['dates'] == AXJ['dates'] and AX['segments'] == AXJ['segments'] == sorted(CTX['seg'].index)
          and AX['dates'][0] == cfg['time']['train_start'], {'segments': len(AX['segments']), 'dates': len(AX['dates']), 'blocks': AX['blocks']})
    COUNTS = {'daily': H19['read_numeric_csv'](DD19 / 'counts_daily.csv.gz', ['segment_id', 'target_date']),
              '6h': H19['read_numeric_csv'](DD19 / 'counts_6h.csv.gz', ['segment_id', 'target_date', 'time_block'])}
    SP = {u: H19['sparse_index'](AX, u, COUNTS[u]) for u in UNITS}
    for u in UNITS:
        check_sparse(SP[u], AX, u)
    check('daily and 6 h sparse targets reconcile', int(SP['daily']['y'].sum()) == int(SP['6h']['y'].sum()), {u: int(SP[u]['y'].sum()) for u in UNITS})
    FOLDS = fold_rows(cfg, {u: AX for u in UNITS})
    T['folds'] = pd.DataFrame(FOLDS)
    check('every fold trains only on targets ending <= first issue - 30 days (research label-latency assumption)',
          all(pd.Timestamp(r['last_train_target_end_local']) <= pd.Timestamp(r['train_cutoff_local']) < pd.Timestamp(r['first_issue_local']) for r in FOLDS), FOLDS[:2])
    nbk = {u: unit_blocks(AX, u) for u in UNITS}
    span = lambda a, b: (pd.Timestamp(b) - pd.Timestamp(a)).days + 1
    f0, f1 = cfg['time']['folds'][0]['start'], cfg['time']['folds'][-1]['end']
    D = cfg['time']['benchmark']['diagnostic_years']
    PRE = DOCS['transfer_preregistration']
    ROWS = {'oof': {u: span(f0, f1) * len(AX['segments']) * nbk[u] for u in UNITS}, 'diag': {u: span(D[0]['start'], D[-1]['end']) * len(AX['segments']) * nbk[u] for u in UNITS},
            'transfer': {u: span(cfg['transfer']['eval_start'], cfg['transfer']['eval_end']) * int(PRE['eval_segments']) * nbk[u] for u in UNITS}}
    check('full OOF populations equal the locked 2022-23 sizes (no missing-context row removal)', ROWS['oof'] == cfg['time']['expected_oof_rows'], ROWS['oof'])
    BUDGET = estimate_artifact_bytes(cfg, ROWS)
    check('estimated new artifacts within the 4 GiB budget (checked before any fit)', BUDGET['within_budget'], BUDGET)
    par = []
    for u in UNITS:
        req = pd.read_csv(DD19 / ('reference_requests_' + u + '.csv'), dtype=str, keep_default_na=False)
        Xc, _ = H19['build_feature_rows'](CTX, req, u, CTX_V)
        Xb, _ = H19['build_feature_rows'](CTX, req, u, CORE_V)
        RL = H19['read_numeric_csv'](DD19 / ('reference_features_' + u + '.csv'), ['segment_id', 'target_date', 'time_block', 'route_code'])
        par.append({'unit': u, 'rows': len(req), 'core_equals_context_prefix_slice': H19['frames_identical'](Xb, Xc[FEATS[u][CORE_V]]),
                    'context_equals_19_reference_features': H19['frames_identical'](Xc, RL[FEATS[u][CTX_V]])})
    T['feature_parity'] = pd.DataFrame(par)
    check('19 builder parity: core variant == leading slice of the context build; context build == accepted 19 reference features (round-trip CSV)',
          all(r['core_equals_context_prefix_slice'] and r['context_equals_19_reference_features'] for r in par), par)
    LOCK = build_lock(cfg, CFG_SHA, PINS, PINS_SHA, INPUT_ROWS.to_dict('records'), {'path': cfg['inputs']['notebook19']['path'], 'functions_sha256': NB19_SHA,
                      'code_cells_sha256': NB19_CELLS}, FEATS, FOLDS, BUDGET, environment())
    LOCK_SHA = write_lock(LOCK, PATHS)
    REPORT_FILES['experiment_lock.json'] = LOCK_SHA
    verify_lock(PATHS, LOCK_SHA)
    check('immutable experiment lock persisted before any fit or score', True, {'lock_sha256': LOCK_SHA})
T['feature_parity']


In [ ]:
# TEMPORAL OOF: per unit and expanding fold -- train-only sample (features built ONCE, reused by every config/variant), every locked component
# fit, exact baselines on the full training calendar, then full-population streamed evaluation of 2022-23 with frozen fold models/baselines.
FIT_RECORDS, SAMPLING, BASE_EVID, OOF, STATUS = [], [], [], {}, {}
for unit in UNITS:
    MODELS, TABS = {}, {}
    for f in cfg['time']['folds']:
        fid = f['fold_id']
        with stage('train_' + unit + '_' + fid):
            T_ = train_dates_per_block(AX, unit, training_cutoff(f['start'], cfg))
            pos, y, w, summ = sample_training(AX, unit, SP[unit], T_, fid, cfg)
            SAMPLING.append(summ)
            Xtr = features_for_positions(CTX, AX, unit, pos, BR)
            check('training sample features built by the 19 builder for every sampled row ' + unit + ' ' + fid, len(Xtr) == len(pos) and list(Xtr.columns) == FEATS[unit][CTX_V], summ)
            MODELS[fid] = {'core': {}, 'context': {}}
            for c in CONFIGS:
                for var, key in ((CORE_V, 'core'), (CTX_V, 'context')):
                    est, rec = fit_component(c, var, Xtr, y, w, FEATS[unit][var], cfg)
                    FIT_RECORDS.append(dict(rec, unit=unit, fold_id=fid))
                    MODELS[fid][key][c['config_id']] = est
                    print('   ', unit, fid, c['config_id'], key, rec['status'], rec['seconds'], 's', flush=True)
                    if est is not None and fid == FOLD_IDS[0]:
                        Xu = Xtr.head(200).copy()
                        Xu['route_code'] = 'UNSEEN_ROUTE_CONTROL'
                        Xm = Xu.copy()
                        for col in FEATS[unit][var]:
                            if col not in cfg['candidates']['categorical_columns']:
                                Xm[col] = np.nan
                        try:
                            ok = bool(np.isfinite(predict_component(est, Xu, FEATS[unit][var], cfg)).all() and np.isfinite(predict_component(est, Xm, FEATS[unit][var], cfg)).all())
                        except Exception as e:
                            ok = False
                        NEG.append({'control': 'unseen_route_and_all_missing_numeric_predict_finite', 'unit': unit, 'method': c['config_id'] + '::' + key, 'passed': ok})
            B = fit_baselines(AX, unit, SP[unit], T_, CTX, cfg)
            TABS[fid], ev = baseline_tables(B, CTX, AX['segments'], cfg)
            BASE_EVID.append(dict(ev, unit=unit, fold_id=fid, training_records=B['training_records'], denominator_cells=B['denominator_cells'],
                                  training_segment_km=B['training_segment_km'], sample_records=summ['training_records']))
            check('baselines use the exact full training calendar (not the sample) ' + unit + ' ' + fid,
                  B['denominator_cells'] == summ['full_training_cells'] and B['training_records'] == summ['training_records'], BASE_EVID[-1])
            del Xtr
    st = {}
    for cid in CIDS:
        core_ok = all(MODELS[f][('core')][cid] is not None for f in FOLD_IDS)
        ctx_ok = all(MODELS[f]['context'][cid] is not None for f in FOLD_IDS)
        st[cid + '::core'], st[cid + '::context'], st[cid + '::routed_context'] = core_ok, ctx_ok, core_ok and ctx_ok
    STATUS[unit] = st
    for f in FOLD_IDS:  # a component that failed in ANY fold is excluded everywhere (no partial pooled metrics)
        for cid in CIDS:
            if not st[cid + '::core']:
                MODELS[f]['core'][cid] = None
            if not st[cid + '::context']:
                MODELS[f]['context'][cid] = None
    save = [cid + '::' + k for cid in CIDS for k in ('core', 'context') if st[cid + '::' + k]] + NONZERO_BASE
    with stage('oof_' + unit):
        periods = [(f['fold_id'], f['start'], f['end']) for f in cfg['time']['folds']]
        OOF[unit] = evaluate_population(CTX, AX, unit, SP[unit], periods, MODELS, TABS, cfg, PATHS['array_dir'], 'oof', save)
    check('full-population OOF ' + unit + ': every cell predicted, arrays complete/finite/non-negative', OOF[unit]['rows'] == cfg['time']['expected_oof_rows'][unit],
          {'rows': OOF[unit]['rows'], 'seconds': OOF[unit]['seconds']})
    y_oof = np.load(PATHS['array_dir'] / ('oof_' + unit + '_y.npy'), mmap_mode='r', allow_pickle=False)
    oof_pos = (AX['date_pos'][f0] * len(AX['segments']) * nbk[unit] + np.arange(0, len(y_oof), 1000003)).astype(np.int64)
    check('saved OOF labels equal the 19 sparse lookups (sampled positions) ' + unit,
          np.array_equal(np.asarray(y_oof[::1000003]), H19['lookup_counts'](SP[unit], oof_pos).astype(float)) and float(np.sum(y_oof)) == float(sum(
              OOF[unit]['period_acc'][f][NONZERO_BASE[0]]['y'][0] for f in FOLD_IDS)), {'rows': len(y_oof)})
    del MODELS
T['model_fits'] = pd.DataFrame(FIT_RECORDS)
T['sampling'] = pd.DataFrame(SAMPLING)
T['model_fits'][['unit', 'fold_id', 'config_id', 'variant', 'status', 'seconds', 'rows']]


In [ ]:
# SELECTION (frozen before any 2024/25 metric): pooled + per-fold OOF metrics, strongest nonzero baseline, deterministic gate, exploratory
# paired 7-day block bootstrap; the selection JSON is written and hashed here and never changed afterwards.
SEL, BOOT, POOLED, LB, FM, ST, DAYS = {}, {}, {}, [], [], [], {}
with stage('selection'):
    for unit in UNITS:
        R = OOF[unit]
        present = [m for m in method_names(cfg) if all(m in R['period_acc'][f] for f in FOLD_IDS)]
        pooled_acc = {m: acc_merge(*[R['period_acc'][f][m] for f in FOLD_IDS]) for m in present}
        POOLED[unit] = {m: acc_metrics(a) for m, a in pooled_acc.items()}
        per_fold = {f: {m: acc_metrics(R['period_acc'][f][m]) for m in present} for f in FOLD_IDS}
        elig = {m: bool(STATUS[unit].get(m)) for m in STATUS[unit] if m.split('::')[1] in cfg['candidates']['eligible_policies']}
        SEL[unit] = select_unit(POOLED[unit], per_fold, elig, cfg)
        for m in present:
            kind = 'baseline' if '::' not in m else m.split('::')[1]
            LB.append(dict(unit=unit, method=m, kind=kind, deployment_eligible=(kind in cfg['candidates']['eligible_policies'] and elig.get(m, False)) or
                           (kind == 'baseline' and m in NONZERO_BASE), **POOLED[unit][m]))
            for f in FOLD_IDS:
                FM.append(dict(unit=unit, fold_id=f, method=m, **per_fold[f][m]))
        ST += strata_rows({m: R['strata'][m] for m in present}, unit, 'oof_2022_2023')
        dates = AX['dates'][AX['date_pos'][f0]:AX['date_pos'][f1] + 1]
        DAYS[unit] = pd.DataFrame({'target_date': dates, **{m: R['day_dev'][m] for m in present}})
        check('pooled OOF deviance equals the sum of the daily contributions ' + unit,
              all(abs(DAYS[unit][m].sum() - pooled_acc[m]['dev'][0]) <= 1e-9 * max(1.0, abs(pooled_acc[m]['dev'][0])) for m in present), {'methods': len(present)})
        bc = SEL[unit].get('best_candidate')
        BOOT[unit] = dict(block_bootstrap(DAYS[unit][bc].to_numpy(), DAYS[unit][SEL[unit]['strongest_baseline']].to_numpy(), cfg), candidate=bc,
                          baseline=SEL[unit]['strongest_baseline']) if bc else {'ci_status': 'no eligible candidate', 'candidate': None, 'baseline': SEL[unit]['strongest_baseline'],
                                                                               'point_relative_deviance_reduction': None, 'ci_low': None, 'ci_high': None}
    SELECTION_DOC = {'kind': 'frozen_selection', 'frozen_utc': utcnow(), 'experiment_lock_sha256': LOCK_SHA, 'frozen_before_2024_2025_metrics': True,
                     'units': SEL, 'bootstrap_exploratory': BOOT, 'component_status': STATUS,
                     'note': 'Selection by pooled 2022-23 full-population OOF deviance and the locked gates; never changed after the previously seen 2024/25 diagnostics.'}
    SEL_SHA = put_data('selection_frozen.json', SELECTION_DOC)
    put_report('selection_frozen.json', SELECTION_DOC)
    check('selection frozen (hashed) before any 2024/25 diagnostic metric', REPORT_FILES['selection_frozen.json'] == SEL_SHA, {'sha256': SEL_SHA})
    for unit in UNITS:
        put_data('oof_daily_deviance_' + unit + '.csv', DAYS[unit])
pd.DataFrame([{'unit': u, 'strongest_baseline': SEL[u]['strongest_baseline'], 'best_candidate': SEL[u].get('best_candidate'), 'recommended': SEL[u]['recommended'],
               'selected': SEL[u]['selected_method'], 'reduction': SEL[u].get('relative_deviance_reduction'), 'mae_ratio': SEL[u].get('mae_ratio'),
               'fold_wins': SEL[u].get('n_fold_wins'), 'reason': SEL[u]['reason']} for u in UNITS])


In [ ]:
# PREVIOUSLY SEEN 2024/25 DIAGNOSTICS (NOT an untouched test): only the frozen selected policy's components + the strongest baseline, fitted once at
# issue 2023-12-31T18:00+07 minus 30 days; full populations per year; saved verified model bundles with reference predictions. Winner never changes.
DIAG, DIAG_ROWS, BUNDLES = {}, [], []
verify_pinned(PATHS['data_dir'] / 'selection_frozen.json', SEL_SHA, 'frozen selection')
for unit in UNITS:
    with stage('diagnostics_' + unit):
        sel, base = SEL[unit]['selected_method'], SEL[unit]['strongest_baseline']
        T_ = train_dates_per_block(AX, unit, training_cutoff(cfg['time']['benchmark']['fit_issue_target_date'], cfg))
        comp = {'core': {}, 'context': {}}
        if '::' in sel:
            cid, policy = sel.split('::')
            conf = next(c for c in CONFIGS if c['config_id'] == cid)
            pos, y, w, summ = sample_training(AX, unit, SP[unit], T_, 'benchmark_2024', cfg)
            SAMPLING.append(summ)
            Xtr = features_for_positions(CTX, AX, unit, pos, BR)
            for var, key in ((CORE_V, 'core'),) + (((CTX_V, 'context'),) if policy == 'routed_context' else ()):
                est, rec = fit_component(conf, var, Xtr, y, w, FEATS[unit][var], cfg)
                FIT_RECORDS.append(dict(rec, unit=unit, fold_id='benchmark_2024'))
                check('benchmark component fitted for the frozen selection ' + unit + ' ' + key, est is not None, rec)
                ref = Xtr.head(1000)
                p_ref = predict_component(est, ref, FEATS[unit][var], cfg)
                path = PATHS['model_dir'] / unit / (cid + '__' + key + '.joblib')
                meta = save_bundle(est, path, {'unit': unit, 'config_id': cid, 'family': conf['family'], 'variant': var, 'columns': FEATS[unit][var],
                                               'fit_scope': 'benchmark 2021..cutoff ' + str(training_cutoff(cfg['time']['benchmark']['fit_issue_target_date'], cfg)),
                                               'experiment_lock_sha256': LOCK_SHA, 'selection_sha256': SEL_SHA})
                est2, _ = load_bundle(path, meta['sha256'])
                same = np.array_equal(predict_component(est2, ref, FEATS[unit][var], cfg), p_ref)
                rk = H19['keys_from_positions'](AX, unit, pos[:1000])
                ref_sha = put_data('models/' + unit + '/' + cid + '__' + key + '.reference.csv', rk.assign(prediction=p_ref))
                BUNDLES.append(dict(meta, reference_predictions_sha256=ref_sha, reload_identical=same))
                check('model bundle reloads only after hash verification and reproduces reference predictions ' + unit + ' ' + key, same, meta['sha256'])
                comp[key][cid] = est
            del Xtr
        B = fit_baselines(AX, unit, SP[unit], T_, CTX, cfg)
        tabs, ev = baseline_tables(B, CTX, AX['segments'], cfg)
        BASE_EVID.append(dict(ev, unit=unit, fold_id='benchmark_2024', training_records=B['training_records'], denominator_cells=B['denominator_cells'],
                              training_segment_km=B['training_segment_km']))
        yrs = cfg['time']['benchmark']['diagnostic_years']
        periods = [(d['name'], d['start'], d['end']) for d in yrs]
        R = evaluate_population(CTX, AX, unit, SP[unit], periods, {p[0]: comp for p in periods}, {p[0]: tabs for p in periods}, cfg, PATHS['array_dir'], 'diag',
                                list(dict.fromkeys([sel, base])))
        DIAG[unit] = []
        for p in periods:
            for m in dict.fromkeys([sel, base]):
                r = dict(unit=unit, year=p[0], method=m, role=('selected' if m == sel else '') + ('+' if m == sel == base else '') + ('strongest_baseline' if m == base else ''),
                         **acc_metrics(R['period_acc'][p[0]][m]))
                DIAG[unit].append(r)
                DIAG_ROWS.append(r)
        ST += strata_rows({m: R['strata'][m] for m in dict.fromkeys([sel, base])}, unit, 'seen_2024_2025')
        OOF[unit]['diag_arrays'] = R['arrays']
        OOF[unit]['diag_periods'] = R['periods']
        OOF[unit]['diag_context_share'] = R['context_share']
verify_pinned(PATHS['data_dir'] / 'selection_frozen.json', SEL_SHA, 'frozen selection (unchanged after diagnostics)')
check('frozen selection unchanged after the 2024/25 diagnostics', True, {'sha256': SEL_SHA})
pd.DataFrame(DIAG_ROWS)[['unit', 'year', 'method', 'role', 'mean_poisson_deviance', 'mae', 'total_pred', 'total_actual']]


In [ ]:
# SPATIAL TRANSFER DIAGNOSTIC (separate from temporal selection): geometry-only manager split recomputed and verified; fixed calendar_geometry
# configs (NOT the temporal winner) trained on the training universe 2021-22 under the 30-day embargo at the first 2023 issue; baselines fit on the
# training universe only (every held-out segment uses the explicit unseen fallback); full 2023 evaluation of the held-out sections.
TRANSFER, TR_ROWS, TR_FITS = {}, [], []
with stage('transfer_split'):
    EVAL_IDS, TRAIN_IDS, SPLIT = transfer_split(CTX['seg'], DD19 / 'universe_geometry.geojson', PRE, cfg['transfer']['id_hash_encodings'])
    check('transfer split reproduces the manager geometry-only preregistration (counts, sorted-id sha, OID disjoint, >= 1 km buffer)', all(SPLIT['checks'].values()), SPLIT)
    check('transfer split uses no labels: eval/train ids disjoint and inside the universe', not (set(EVAL_IDS) & set(TRAIN_IDS)) and set(EVAL_IDS) | set(TRAIN_IDS) <= set(AX['segments']),
          {'eval': len(EVAL_IDS), 'train': len(TRAIN_IDS)})
tc = cfg['transfer']
for unit in UNITS:
    with stage('transfer_' + unit):
        C = COUNTS[unit]
        ax_tr = H19['population_axes'](TRAIN_IDS, AX['dates'][0], AX['dates'][-1], AX['blocks'])
        sp_tr = H19['sparse_index'](ax_tr, unit, C[C['segment_id'].isin(set(TRAIN_IDS))].reset_index(drop=True))
        ax_ev = H19['population_axes'](EVAL_IDS, tc['eval_start'], tc['eval_end'], AX['blocks'])
        ce = C[C['segment_id'].isin(set(EVAL_IDS)) & (C['target_date'] >= tc['eval_start']) & (C['target_date'] <= tc['eval_end'])].reset_index(drop=True)
        sp_ev = H19['sparse_index'](ax_ev, unit, ce)
        T_ = train_dates_per_block(ax_tr, unit, training_cutoff(tc['first_issue_target_date'], cfg))
        pos, y, w, summ = sample_training(ax_tr, unit, sp_tr, T_, 'transfer_2023', cfg)
        SAMPLING.append(dict(summ, scope='transfer_training_universe'))
        Xtr = features_for_positions(CTX, ax_tr, unit, pos, BR)
        comp = {'core': {}, 'context': {}}
        for cid in tc['configs']:
            conf = next(c for c in CONFIGS if c['config_id'] == cid)
            est, rec = fit_component(conf, tc['variant'], Xtr, y, w, FEATS[unit][tc['variant']], cfg)
            TR_FITS.append(dict(rec, unit=unit, fold_id='transfer_2023'))
            comp['core'][cid] = est
        del Xtr
        B = fit_baselines(ax_tr, unit, sp_tr, T_, CTX, cfg)
        tabs, ev = baseline_tables(B, CTX, ax_ev['segments'], cfg)
        check('transfer baselines: every held-out segment uses the explicit unseen-segment fallback ' + unit, ev['unseen_segments'] == len(EVAL_IDS), ev)
        save = [cid + '::core' for cid in tc['configs'] if comp['core'][cid] is not None] + NONZERO_BASE
        R = evaluate_population(CTX, ax_ev, unit, sp_ev, [('transfer_2023', tc['eval_start'], tc['eval_end'])], {'transfer_2023': comp}, {'transfer_2023': tabs},
                                cfg, PATHS['array_dir'] / 'transfer', 'transfer', save, save_mask=False)
        TRANSFER[unit] = {'rows': R['rows'], 'arrays': R['arrays'], 'baseline_fallbacks': ev, 'sampling': summ, 'seconds': R['seconds'],
                          'model_status': {r['config_id']: r['status'] for r in TR_FITS if r['unit'] == unit}}
        for m, a in R['period_acc']['transfer_2023'].items():
            TR_ROWS.append(dict(unit=unit, method=m, kind='baseline' if '::' not in m else 'fixed_core_config', **acc_metrics(a)))
T['transfer'] = pd.DataFrame(TR_ROWS)
T['transfer'][['unit', 'method', 'mean_poisson_deviance', 'mae', 'total_pred', 'total_actual']]


In [ ]:
# REPORTS + MANIFEST: aggregate CSVs (lock sha stamped), actual-result figures, Thai narrative generated from the computed objects, negative
# controls, array/model hashes, protected-before/after and the strict manifest. Row-level arrays/models stay in the ignored data dir.
with stage('reports'):
    LBd, FMd, STd = pd.DataFrame(LB), pd.DataFrame(FM), pd.DataFrame(ST)
    abl = []
    for u in UNITS:
        for cid in CIDS:
            r = {'unit': u, 'config_id': cid}
            for k in ('core', 'context', 'routed_context'):
                m = POOLED[u].get(cid + '::' + k)
                r[k + '_deviance'] = m['mean_poisson_deviance'] if m else None
                r[k + '_mae'] = m['mae'] if m else None
            abl.append(r)
    for r in FOLDS:
        NEG.append({'control': 'embargo_target_end_le_issue_minus_30d', 'unit': r['unit'], 'method': r['fold_id'],
                    'passed': pd.Timestamp(r['last_train_target_end_local']) <= pd.Timestamp(r['train_cutoff_local'])})
    for s in SAMPLING:
        NEG.append({'control': 'sample_has_no_future_or_duplicate_cells', 'unit': s['unit'], 'method': s['fold_id'], 'passed': bool(s['no_future_cells'] and s['unique_positions'])})
    for bad in ('segment_id', 'y_count', 'event_lat', 'target_date', 'oid'):
        try:
            H19['validate_columns'](FEATS['6h'][CTX_V] + [bad], CTX['settings'])
            ok = False
        except ValueError:
            ok = True
        NEG.append({'control': 'predictor_allowlist_rejects_' + bad, 'unit': 'both', 'method': 'validate_columns', 'passed': ok})
    NEG += routed_trigger_controls(cfg, FEATS, CTX['settings'])
    NEGd = pd.DataFrame(NEG)
    check('all negative controls pass', bool(NEGd['passed'].all()), NEGd[~NEGd['passed']].to_dict('records'))
    AV = pd.DataFrame([dict(unit=u, period=p, **v) for u in UNITS for p, v in {**OOF[u]['context_share'], **OOF[u].get('diag_context_share', {})}.items()])
    SELd = pd.DataFrame([dict(unit=u, strongest_baseline=SEL[u]['strongest_baseline'], best_candidate=SEL[u].get('best_candidate'), recommended=SEL[u]['recommended'],
                              selected_method=SEL[u]['selected_method'], relative_deviance_reduction=SEL[u].get('relative_deviance_reduction'), mae_ratio=SEL[u].get('mae_ratio'),
                              n_fold_wins=SEL[u].get('n_fold_wins'), reason=SEL[u]['reason'], **{'best_' + k: v for k, v in SEL[u]['best_per_family'].items()}) for u in UNITS])
    BOOTd = pd.DataFrame([dict(unit=u, **BOOT[u]) for u in UNITS])
    for name, df in (('leaderboard_oof.csv', LBd), ('fold_metrics.csv', FMd), ('selection.csv', SELd), ('feature_ablation.csv', pd.DataFrame(abl)),
                     ('diagnostics_seen_2024_2025.csv', pd.DataFrame(DIAG_ROWS)), ('transfer_metrics.csv', T['transfer']),
                     ('transfer_split.csv', pd.DataFrame([{k: v for k, v in SPLIT.items() if k not in ('checks', 'id_hash_match_by_encoding')}])),
                     ('sampling.csv', pd.DataFrame(SAMPLING)), ('context_availability.csv', AV), ('negative_controls.csv', NEGd), ('strata_metrics.csv', STd),
                     ('model_fits.csv', pd.DataFrame(FIT_RECORDS + TR_FITS).assign(warnings=lambda d: d['warnings'].map(lambda v: ' | '.join(v)))),
                     ('bootstrap.csv', BOOTd), ('baseline_fallbacks.csv', pd.DataFrame(BASE_EVID)), ('folds.csv', T['folds'].astype(str)),
                     ('feature_parity.csv', T['feature_parity']), ('inputs.csv', INPUT_ROWS)):
        put_report(name, df)
    set_fonts(cfg)
    CAL = STd[(STd['stratum'] == 'pred_bin') & (STd['scope'] == 'oof_2022_2023')]
    FIGS = make_figures(PATHS['figure_dir'], LBd, FMd, CAL, T['transfer'], SEL)
    for fn in FIGS:
        REPORT_FILES['figures/' + fn] = sha256_file(PATHS['figure_dir'] / fn)
    check('at least three actual-result figures written', len(FIGS) >= 3, FIGS)
    put_report('results_th.md', narrative_th(SEL, BOOT, {u: DIAG[u] for u in UNITS}, {u: [r for r in TR_ROWS if r['unit'] == u] for u in UNITS}, LOCK_SHA).encode('utf-8'))
    ARRAYS = {u: {'oof': OOF[u]['arrays'], 'oof_periods': OOF[u]['periods'], 'diag': OOF[u].get('diag_arrays'), 'diag_periods': OOF[u].get('diag_periods'),
                  'transfer': TRANSFER[u]['arrays']} for u in UNITS}
    put_data('arrays_index.json', {'experiment_lock_sha256': LOCK_SHA, 'order': 'date, then segment_id (sorted), then time_block; implicit axes = 19 population_axes.json',
                                   'zero_baseline': 'not stored (identically 0)', 'routed_context': 'derive as where(context_mask == 1, <config>__context, <config>__core)',
                                   'transfer_axes': {'eval_segments': EVAL_IDS, 'dates': [cfg['transfer']['eval_start'], cfg['transfer']['eval_end']]}, 'units': ARRAYS})
    put_data('model_bundles.json', {'experiment_lock_sha256': LOCK_SHA, 'bundles': BUNDLES})
    verify_lock(PATHS, LOCK_SHA)
    check('experiment lock unchanged at the end of the run', True, LOCK_SHA)
    DATA_FILES = {q.relative_to(PATHS['data_dir']).as_posix(): sha256_file(q) for q in sorted(PATHS['data_dir'].rglob('*')) if q.is_file()}
    new_bytes = sum((PATHS['data_dir'] / k).stat().st_size for k in DATA_FILES)
    check('new data artifacts within the 4 GiB budget', new_bytes <= int(cfg['resources']['artifact_budget_bytes']), {'bytes': new_bytes, 'estimate': BUDGET['total_bytes']})
    PROTECTED_AFTER = hash_paths(BACKEND, protected_rels(cfg))
    check('protected V2/V3 inputs, configs, notebooks and protocols unchanged', PROTECTED_AFTER == PROTECTED_BEFORE, {'files': len(PROTECTED_BEFORE)})
    check('source config unchanged during the run', sha256_file(CONFIG_PATH) == CFG_SHA)
    failed_models = [r for r in FIT_RECORDS + TR_FITS if not str(r['status']).startswith('ok')]
    EXTRA = {'notebook19_functions_sha256': NB19_SHA, 'manager_pins_sha256': PINS_SHA, 'selection_sha256': SEL_SHA, 'selection': SEL, 'bootstrap': BOOT,
             'transfer': {'split': SPLIT, **{u: {k: v for k, v in TRANSFER[u].items() if k != 'arrays'} for u in UNITS}},
             'failed_models': failed_models, 'convergence_warnings': [r for r in FIT_RECORDS + TR_FITS if r.get('convergence_warning')],
             'durations': DURATIONS, 'peak_rss_bytes': peak_rss_bytes(), 'artifact_bytes': new_bytes, 'artifact_budget': BUDGET,
             'figures': FIGS, 'oof_rows': {u: OOF[u]['rows'] for u in UNITS}}
    MANIFEST = build_manifest('complete', STARTED, cfg, CFG_SHA, LOCK_SHA, NB_PATH, CHECKS, REPORT_FILES, DATA_FILES, PROTECTED_BEFORE, PROTECTED_AFTER, EXTRA)
    atomic_write_json(PATHS['report_dir'] / 'spatial_experiments_manifest.json', MANIFEST)
    read_strict_json(PATHS['report_dir'] / 'spatial_experiments_manifest.json')
print('status', MANIFEST['status'], '| failed checks', MANIFEST['failed_checks'], '| failed models', len(failed_models))
SELd


## Outputs

- `reports/v3/spatial_experiments/` (tracked, aggregate only): `experiment_lock.json`, `selection_frozen.json`, `spatial_experiments_manifest.json`, CSVs (`leaderboard_oof`, `fold_metrics`, `selection`, `feature_ablation`, `diagnostics_seen_2024_2025`, `transfer_metrics`, `transfer_split`, `sampling`, `context_availability`, `negative_controls`, `strata_metrics`, `model_fits`, `bootstrap`, `baseline_fallbacks`, `folds`, `feature_parity`, `inputs`), `figures/*.png`, `results_th.md`.
- `data/v3/spatial_experiments/` (gitignored, row level): `arrays/oof_<unit>_<method>.npy`, `oof_<unit>_y.npy`, `oof_<unit>_context_mask.npy`, `diag_*`, `arrays/transfer/transfer_*` (float64 / uint8 `.npy`, load with `np.load(path, mmap_mode='r', allow_pickle=False)`; implicit date → segment → block order, offsets in `arrays_index.json`), `oof_daily_deviance_<unit>.csv`, `selection_frozen.json`, `models/<unit>/*.joblib` (+ `.meta.json`, load only through `load_bundle` after sha verification) and `model_bundles.json`.

All metrics can be recomputed from the saved arrays: `routed_context = where(mask == 1, context, core)`; deviance floors predictions at 1e-12 only inside the metric.
